# BatSpot Fine-Tuning on Kaggle — Detector + Classifier Pipeline

Fine-tune BatSpot pre-trained models on custom bat vocalisation data.

**Pipeline:**
1. **Setup** — Config, dependencies, model/dataset definitions
2. **Data Prep** — Load data, create detector (binary) and classifier (8-class) datasets; recording-leakage check; class-prior report
3. **Pretrained Models** — Load detector.pk (192kHz) + classifier.pk (250kHz) from Kaggle upload
4. **Detector Training** — Fine-tune binary detector(s), evaluate
5. **Classifier Training** — Fine-tune 8-class classifier, evaluate
6. **Combined Pipeline** — Classifier alone vs detector hard-gate (raw and prior-corrected) vs soft combination
7. **Export** — Save the .pk files
8. **Existing Model Evaluation** — Official models scored zero-shot on the same test clips
9. **Extra Evidence** *(added by Space Bunny Free)* — augmentation A/B, seed spread, recording-grouped retrain

**Input windowing (important).** Clips are ~375 ms (median) but the CNN reads 20 ms. Training uses a
fresh random 20 ms crop from the loudest part of each clip every epoch; validation/test average the
top-5 loudest windows (overlap < half a window). The earlier version used only the *first* 20 ms of
every clip, which capped accuracy (classifier ~0.75, detector ~0.85); see `AGENTS.md` ("Windowing fix")
for the measurements. Set `WINDOW_MODE = 'first'` in Cell 2 to reproduce the old window choice.

---

## What changed relative to `batspot-train.ipynb`

Base notebook: `batspot-train.ipynb`. Cells 3, 5, 7, 8, 10, 11, 12, 15, 16 and 19 are byte-identical.
Full reasoning, evidence and the review of the base notebook are in `AGENTS.md` §9.

### Correctness fixes (each was a live latent bug)

| # | Cell | Bug | Effect |
| :-- | :--- | :--- | :--- |
| 1 | 9 | `save_path` gated **both** the best-weights snapshot and its restore | With `save_path=None` (the signature default) `train_model` returned **last-epoch** weights while still reporting the **best** validation score. A plot filename decided which model you got. |
| 2 | 6 | `minmax_normalize(pad_window(...))` — padded **before** normalising | A literal `0.0` inserted into a dB array is normally the array *maximum*, so min-max mapped the padded region to **1.0** — silence shown to the model as the loudest thing in the window. Official order is normalise → pad. |
| 3 | 2, 9 | `val_bal if SELECT_METRIC == 'balanced_accuracy' else val_acc` | Any typo silently fell back to plain accuracy, reinstating the exact all-call bug the metric switch exists to prevent — with no diagnostic, while the log still printed the metric name. |
| 4 | 7 | `recording_id()` indexed `parts[3]` of a 6-field filename | Cell 8 writes **4-field** names, so clips from this notebook's own extractor were each their own "recording". The leakage check printed a **false all-clear**, and `SPLIT_BY_RECORDING=True` became a random split while claiming to hold out tapes. |
| 5 | 17, 18 | Summary table dropped the `kind` column | The official classifier's row read "acc 0.8069" with nothing saying that is 117/145 (it answers "not noise" for every clip) and its AUC is 0.4924 — **no signal**. The misreading §8.2 item 4 was written to prevent, reintroduced one cell later. |

Also hardened: spectrogram cache is now keyed on the source directory and validated against the
wav's size+mtime (it was `os.path.exists`, so re-extracted clips silently reused old
spectrograms); the cache version tag is derived from the front-end source instead of hand-bumped;
`np.save` is atomic; a failing clip is reported and dropped instead of killing the run; NaN
spectrograms are scrubbed; `train_window_starts` takes an exact top-k count (percentile ties were
selecting *every* window on 3 clips); `predict_proba` refuses to return an all-zero row; class
indices in Cell 17 are resolved by name with a `num_mels` fallback.

### Additions

| Cell | What | Why |
| :--- | :--- | :--- |
| 9 | `REPORT_TOPK_MEAN` — prints the mean of the top-3 validation scores beside the argmax | The returned checkpoint is the argmax over up to 120 validations of a *discrete* metric on 145 clips; one clip is worth 0.0221. This turns "should `n_epochs` be larger?" from an argument into a measurement. Also writes the training curves that `save_path` always promised. |
| 7, 14 | Class priors reported; detector gate evaluated **raw and prior-corrected** | The sampler forces a 50/50 train prior against an ~19/81 eval prior, a ~4.2× shift in odds. That is why m09 showed noise **precision 0.70 vs recall 0.93** and why gate thresholds sank to 0.30/0.40/0.10. Such a threshold encodes the test noise fraction and will not transfer to real recordings, where noise is >95 % of windows. |
| 7 | Recordings-per-class table | Shows directly which classes *can* be validated on a new tape (heti: 1) and which cannot. |
| 20 | Augmentation A/B + seeds 42/43/44 | Gives the ± spread needed before any 1–3 point difference in this notebook can be believed. |
| 21 | Retrain with whole tapes held out | The only honest new-recording estimate; the in-split number cannot be converted into one. |

**Expect the main results to be close to the base notebook.** The bug fixes are latent on this
dataset (0 of 964 clips are shorter than one window; `save_path` is always passed; `SELECT_METRIC`
is spelled correctly; the Kaggle clips use 6-field names). The prior-corrected gate column and the
two new cells are the parts that will differ.

**Read `AGENTS.md` §9.7 before trusting any single number.** These are 145-clip test sets.


In [ ]:
# Cell 2: Configuration
import os

# --- Paths (Kaggle defaults, override as needed) ---
DATA_DIR = '/kaggle/input/datasets/budhil/100eachbatds/Data/Data/final_dataset/data'  # Pre-extracted clips (class subfolders)
RAW_AUDIO_DIR = '/kaggle/input/datasets/budhil/100eachbatds/Data/Data/audio'                                         # Raw .WAV recordings (for clip extraction)
SELECTIONS_DIR = '/kaggle/input/datasets/budhil/100eachbatds/Data/Data/selections'                                        # Raven selection tables (for clip extraction)

# Clip extraction re-reads ALL raw audio and writes into DATA_DIR. You already
# have the extracted clips in final_dataset/data, so leave this False unless you
# specifically want to rebuild them from the raw recordings.
RUN_CLIP_EXTRACTION = True
OVERWRITE_EXISTING_CLIPS = False   # guard: refuse to write into a non-empty DATA_DIR
MODEL_OUTPUT_DIR = None                                      # Your own previously-trained models (validation)

# --- Auto-detect MODEL_OUTPUT_DIR relative to DATA_DIR ---
if MODEL_OUTPUT_DIR is None and DATA_DIR is not None:
    _data_parent = os.path.dirname(os.path.abspath(DATA_DIR))
    for _candidate in [
        os.path.join(_data_parent, 'model_output'),
        os.path.join(_data_parent, '..', 'model_output'),
        os.path.join(_data_parent, '..', '..', 'model_output'),
        'Data/model_output',
    ]:
        _candidate = os.path.normpath(_candidate)
        if os.path.isdir(_candidate):
            MODEL_OUTPUT_DIR = _candidate
            break

# ---------------------------------------------------------------------------
# OFFICIAL PRE-TRAINED MODELS -- set paths manually here.
#
# Fill in the 3 call detectors + 1 call classifier you uploaded as a Kaggle
# dataset. Leave any of them as None to skip it (Cell 10 falls back to
# auto-discovery under /kaggle/input/*, then to Data/model_output/).
#
# Click "Add Data" in Kaggle, then use "Copy Path" on the dataset folder to
# get the exact /kaggle/input/... prefix.
#
#   Detector   192 kHz, 2 classes (noise/target)  -> fine-tunes the detector
#   Classifier 250 kHz, 15 classes                -> fine-tunes the classifier
# ---------------------------------------------------------------------------
DETECTOR_M03_PATH = '/kaggle/input/datasets/budhil/100eachbatds/Batspot_Models/Batspot_Models/models_call_detector/m03/ANIMAL-SPOT.pk'   # e.g. '/kaggle/input/mymodels/batspot/models_call_detector/m03/train/ANIMAL-SPOT.pk'
DETECTOR_M09_PATH = '/kaggle/input/datasets/budhil/100eachbatds/Batspot_Models/Batspot_Models/models_call_detector/m09/ANIMAL-SPOT.pk'    # e.g. '/kaggle/input/mymodels/batspot/models_call_detector/m09/train/ANIMAL-SPOT.pk'
DETECTOR_M11_PATH = '/kaggle/input/datasets/budhil/100eachbatds/Batspot_Models/Batspot_Models/models_call_detector/m11/ANIMAL-SPOT.pk'    # e.g. '/kaggle/input/mymodels/batspot/models_call_detector/m11/train/ANIMAL-SPOT.pk'
CLASSIFIER_M09_PATH = '/kaggle/input/datasets/budhil/100eachbatds/Batspot_Models/Batspot_Models/models_call_classifier/ANIMAL-SPOT.pk'  # e.g. '/kaggle/input/mymodels/batspot/models_call_classifier/m09/train/ANIMAL-SPOT.pk'

# Manual paths take priority over auto-discovery. Set to False to ignore them.
USE_MANUAL_MODEL_PATHS = True

WORKING_DIR = '/kaggle/working'
PRETRAINED_DIR = os.path.join(WORKING_DIR, 'pretrained')
os.makedirs(PRETRAINED_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Spectrogram params below are the REAL values read out of the official
# BatSpot .pk files (BatSpot_article/batspot/models_*/**/ANIMAL-SPOT.pk).
# They MUST match the model you fine-tune from, otherwise the transferred
# encoder weights were trained on a different time-frequency resolution and
# are effectively noise. Cell 10 re-verifies this against each .pk at runtime.
# ---------------------------------------------------------------------------

# --- Detector config (192 kHz, binary: noise vs target) ---
DET_CONFIG = {
    'sr': 192000,
    'num_classes': 2,
    'classes': ['noise', 'target'],
    'fmin': 1000,               # from official detector dataOpts
    'fmax': 95000,
    'n_fft': 256,
    'hop_length': 128,
    'n_freq_bins': 256,
    'sequence_len': 20,         # ms
    'max_pool': 2,
    'freq_compression': 'linear',
    # Batch 32 (official: 16). 674 clips / 32 = 22 optimizer steps per epoch. Batch 128 gave
    # only 6 steps/epoch, which is why the old config needed absurd learning rates.
    'batch_size': 32,
    'base_lr': 1e-4,            # absolute LR (official value) — NOT scaled by batch size
    # Cap only. The Kaggle run stopped m09 at the old cap of 60 with its best epoch at 58.
    # NOTE (Space Bunny Free): that is a *weak* justification. A regression of m09's validation
    # balanced accuracy over epochs 20-60 gives +0.00094/epoch (t=+2.09, marginal); over epochs
    # 45-60 it gives +0.00050/epoch (t=+0.43, i.e. nothing). Extrapolated over the extra 40
    # epochs that is +0.02 to +0.06 balanced accuracy = 1 to 3 clips on a 145-clip val set,
    # where one clip is worth 0.0221. So the expected gain is at the measurement limit.
    # It is harmless and may help, but REPORT_TOPK_MEAN below is what will actually settle it.
    'n_epochs': 100,
    'epochs_per_eval': 1,       # validation is cheap (clip-level, top-k windows)
    'grad_accum_steps': 1,
    'use_multi_gpu': True,
    # reference hyperparams from models_call_detector/*/config_training_gui
    'beta1': 0.5,
    'lr_decay_factor': 0.5,
    'lr_patience_epochs': 12,              # RAW epochs without improvement before halving LR
    'early_stopping_patience_epochs': 30,  # RAW epochs without improvement before stopping
}

# --- Classifier config (250 kHz, multi-species) ---
CLS_CONFIG = {
    'sr': 250000,
    'num_classes': None,        # auto-detected from your data in Cell 7
    'fmin': 10000,              # from official classifier dataOpts
    'fmax': 125000,
    'n_fft': 256,
    'hop_length': 128,
    'n_freq_bins': 256,
    'sequence_len': 20,         # ms
    'max_pool': 2,
    'freq_compression': 'linear',
    'batch_size': 32,           # 22 steps/epoch (batch 128 = 6 steps/epoch was the real problem)
    'base_lr': 3e-4,            # measured best of {1e-4, 3e-4, 1e-3} (3 seeds each); 1e-2 is NOT needed
                                # and destabilises training (val acc collapsed to 0.42 at epoch 38).
                                # CAVEAT: that sweep (gpuexp.py) selected checkpoints on plain
                                # accuracy and had no LR schedule and no early stopping, so it is
                                # not the same training loop as this one. See AGENTS.md 9.4.
    'n_epochs': 120,            # val score was still rising at epoch 60 with lr 3e-4
    'epochs_per_eval': 1,
    'grad_accum_steps': 1,
    'use_multi_gpu': True,
    # reference hyperparams from models_call_classifier/m09/config_train
    'beta1': 0.5,
    'lr_decay_factor': 0.5,
    'lr_patience_epochs': 15,              # RAW epochs without improvement before halving LR
    'early_stopping_patience_epochs': 40,  # RAW epochs without improvement before stopping
}

# --- Input windowing (THE main accuracy fix; see AGENTS.md "Windowing fix") ---------------
# Clips are 15 ms - 2.6 s (median ~375 ms) but every model consumes a 20 ms window.
#   'energy_crop' : TRAIN  = a fresh random crop each epoch from the loudest 20 % of the clip's windows
#                   TEST/VAL = the TEST_TOPK loudest windows (overlap < half a window), probs averaged
#   'first'       : old behaviour (only the first 20 ms of every clip) -- kept for ablation
# Measured effect of this single change (AGENTS.md 8.5): classifier 0.754 -> 0.853,
# detector m09 0.866 -> 0.933. It multiplies distinct training inputs by ~x56 (classifier)
# and ~x43 (detector); that is why train loss no longer collapses to ~0.
WINDOW_MODE = 'energy_crop'
WINDOW_STRIDE = 3          # frames between candidate windows (3 frames ~ 2 ms)
TRAIN_TOP_FRAC = 0.20      # train crops come from this loudest fraction of windows
TEST_TOPK = 5              # windows averaged per clip at validation / test time

# Model selection, LR schedule and early stopping all track this validation score.
# 'balanced_accuracy' (mean per-class recall) cannot be gamed by predicting the majority class;
# the detector's 80/20 call/noise split made plain accuracy hide the all-call collapse of m03.
#
# FIX (Space Bunny Free): this used to be read with
#     val_score = val_bal if SELECT_METRIC == 'balanced_accuracy' else val_acc
# which silently falls back to plain accuracy for ANY other string -- a one-character typo
# ('balanced_acc') reinstated the entire bug the metric switch was meant to fix, with no
# diagnostic, while the log still printed "selection metric: balanced_acc". It is now
# validated here, at config time, and selected through a dict lookup.
SELECT_METRIC = 'balanced_accuracy'   # 'balanced_accuracy' | 'accuracy'
assert SELECT_METRIC in ('balanced_accuracy', 'accuracy'), (
    f'SELECT_METRIC must be "balanced_accuracy" or "accuracy", got {SELECT_METRIC!r}. '
    'An unrecognised value used to fall back to plain accuracy without warning.')

# The filename carries the recording (tape) id and there are only ~28 recordings, so a random
# clip-level split leaks recordings between train and test (Cell 7 prints a leakage check).
# True = hold out whole recordings (stricter, noisier, and classes recorded once end up in one split).
SPLIT_BY_RECORDING = False

# --- Augmentation toggle (main path) ---
# True  = time shift + gaussian noise + freq/time masking applied to train windows
# False = no extra augmentation. Random window cropping is already the augmentation that
#         matters here; the extra ones were never shown to help on ~84 clips/class.
# The A/B test of this flag now runs automatically in the ablation cell (Cell 20).
USE_AUGMENTATION = False

# ===========================================================================
# ADDED BY SPACE_BUNNY_FREE  (see AGENTS.md section 9 for the reasoning)
# ===========================================================================

# Report the mean of the top-k validation scores next to the argmax.
# WHY: the returned checkpoint is the single best of up to 120 validations of a DISCRETE
# metric on 145 clips. One flipped clip moves balanced accuracy by 0.0221, so the argmax is
# biased upward by roughly 2 clips relative to the run's typical validation level. Printing
# the top-k mean next to the argmax makes that optimism visible instead of arguing about it.
REPORT_TOPK_MEAN = 3

# Correct the detector gate for the train/eval class-prior mismatch.
# WHY: make_weighted_sampler draws each class with probability exactly 1/n_classes, so the
# detector is trained under a 50/50 noise/call prior while the val/test split is 20/80. The
# resulting probabilities are shifted towards 'call' by a factor (0.807/0.5)/(0.193/0.5) = 4.2
# in odds, which is why m09 shows noise PRECISION 0.70 against recall 0.93 and why the
# val-tuned gate thresholds had to drop to 0.30/0.40/0.10. Those thresholds fit the 20 % noise
# prior and will not transfer to real recordings, where noise is >95 % of windows. Rescaling by
# prior_eval/prior_train makes 0.5 mean what it says.
# Both raw and corrected thresholds are printed, so the Kaggle run stays comparable.
PRIOR_CORRECT_GATE = True

# Classifier A/B ablations + multi-seed spread (runs in Cell 20, after the main models are
# trained and exported, so it cannot disturb them). Each seed re-splits AND re-initialises, so
# the reported sd is total run-to-run spread. This is the number needed before any 1-3 point
# difference in this notebook can be believed.
RUN_ABLATIONS = True
ABLATION_SEEDS = [42, 43, 44]

# Recording-grouped retrain (Cell 21): the only honest estimate of performance on a NEW
# recording. It cannot be derived from the in-split numbers -- the leakage check reports
# 145/145 test clips share a recording with train, so there is no clean subset to score.
RUN_GROUPED_CHECK = True
GROUPED_MICS = ['m09']        # best mic only; retraining all three triples the cost for no extra insight

# A model whose AUC sits within this of 0.5 carries no usable signal and is labelled as such
# in the summary tables, so a high plain accuracy on an 80/20 split cannot be misread.
AUC_NO_SIGNAL = 0.05

# --- GPU check ---
# If DEVICE prints 'cpu', the Kaggle GPU accelerator is OFF.
# To enable: Session options (right sidebar) -> Accelerator -> GPU T4 x2 -> Save.
# The notebook must then be restarted. No code change is needed.
DEVICE = __import__('torch').device('cuda' if __import__('torch').cuda.is_available() else 'cpu')

print(f'Detector:   {DET_CONFIG["sr"]//1000}kHz, {DET_CONFIG["num_classes"]} classes '
      f'(noise/target), fmin={DET_CONFIG["fmin"]}, fmax={DET_CONFIG["fmax"]}')
print(f'Classifier: {CLS_CONFIG["sr"]//1000}kHz, {CLS_CONFIG["num_classes"]} classes (auto), '
      f'fmin={CLS_CONFIG["fmin"]}, fmax={CLS_CONFIG["fmax"]}')
print(f'Device: {DEVICE}')
print(f'Selection metric: {SELECT_METRIC} (validated)   top-{REPORT_TOPK_MEAN} mean reported: '
      f'{bool(REPORT_TOPK_MEAN)}   prior-corrected gate: {bool(PRIOR_CORRECT_GATE)}')
print(f'Extra runs enabled -> ablations: {bool(RUN_ABLATIONS)} {ABLATION_SEEDS}, '
      f'grouped check: {bool(RUN_GROUPED_CHECK)} {GROUPED_MICS}')

# Report which manual paths resolved (a typo here is otherwise silent)
if USE_MANUAL_MODEL_PATHS:
    print('\nManual pre-trained model paths:')
    for _lbl, _p in [('detector m03', DETECTOR_M03_PATH),
                     ('detector m09', DETECTOR_M09_PATH),
                     ('detector m11', DETECTOR_M11_PATH),
                     ('classifier m09', CLASSIFIER_M09_PATH)]:
        if _p is None:
            print(f'  {_lbl:15s} (not set)')
        elif os.path.isfile(_p):
            print(f'  {_lbl:15s} OK  {_p}')
        else:
            print(f'  {_lbl:15s} !! NOT FOUND: {_p}')
else:
    print('\nManual model paths DISABLED (USE_MANUAL_MODEL_PATHS = False)')


In [ ]:
# Cell 3: Install dependencies
!pip install -q resampy soundfile tqdm

import resource
try:
    resource.setrlimit(resource.RLIMIT_NOFILE, (65535, 65535))
except Exception:
    pass

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} ({props.total_memory / 1e9:.1f} GB)')


In [ ]:
# Cell 4: Imports
import os, sys, glob, json, shutil, zipfile, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from collections import OrderedDict, defaultdict, Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, roc_auc_score
)
import matplotlib.pyplot as plt
import resampy
import soundfile as sf
from tqdm.auto import tqdm
from math import ceil

sys.path.insert(0, os.path.abspath('.'))

print('All imports OK')

In [ ]:
# Cell 5: Model Architecture & Loading
# --- Faithful port of animal_spot/models/ (GPL-3.0, Bergler & Schroeter) ---
# Module/attribute names MUST match the original, or saved .pk state_dicts
# will not load. In particular the residual projection is `shortcut`
# (ResidualBase assigns self.shortcut = downsample), NOT `downsample`.

def get_padding(kernel_size):
    """Return `same` padding for a given kernel size."""
    if isinstance(kernel_size, int):
        return kernel_size // 2
    return tuple(s // 2 for s in kernel_size)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_ch, out_ch, stride=1, downsample=None, upsample=None, mid_ch=None):
        super().__init__()
        if mid_ch is None:
            mid_ch = out_ch
        if downsample is not None and upsample is not None:
            raise ValueError("Either downsample or upsample has to be None")

        if upsample is None:
            self.shortcut = downsample
            self.conv1 = nn.Conv2d(in_ch, mid_ch, kernel_size=3, stride=stride,
                                   padding=get_padding(3), bias=False)
        else:
            self.shortcut = upsample
            self.conv1 = nn.ConvTranspose2d(in_ch, mid_ch, kernel_size=3, stride=stride,
                                            padding=get_padding(3),
                                            output_padding=get_padding(stride), bias=False)
        self.bn1 = nn.BatchNorm2d(mid_ch)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(mid_ch, out_ch, kernel_size=3, stride=1,
                               padding=get_padding(3), bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu2 = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.shortcut is not None:
            residual = self.shortcut(x)
        out += residual
        out = self.relu2(out)
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_ch, out_ch, stride=1, downsample=None, upsample=None, mid_ch=None):
        super().__init__()
        if mid_ch is None:
            mid_ch = out_ch
        if downsample is not None and upsample is not None:
            raise ValueError("Either downsample or upsample has to be None")
        self.shortcut = None
        if upsample is not None or downsample is not None:
            self.shortcut = downsample if downsample is not None else upsample

        self.conv1 = nn.Conv2d(in_ch, mid_ch, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_ch)
        self.relu1 = nn.ReLU(inplace=True)
        if upsample is not None:
            self.conv2 = nn.ConvTranspose2d(mid_ch, mid_ch, kernel_size=3, stride=stride,
                                            padding=get_padding(3),
                                            output_padding=get_padding(stride), bias=False)
        else:
            self.conv2 = nn.Conv2d(mid_ch, mid_ch, kernel_size=3, stride=stride,
                                   padding=get_padding(3), bias=False)
        self.bn2 = nn.BatchNorm2d(mid_ch)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv3 = nn.Conv2d(mid_ch, out_ch * self.expansion, kernel_size=1,
                               stride=1, padding=get_padding(1), bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch * self.expansion)
        self.relu3 = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.relu1(self.bn1(self.conv1(x)))
        out = self.relu2(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.shortcut is not None:
            residual = self.shortcut(x)
        out += residual
        out = self.relu3(out)
        return out


def get_block_sizes(resnet_size):
    if resnet_size == 18:
        return [2, 2, 2, 2]
    elif resnet_size == 34:
        return [3, 4, 6, 3]
    elif resnet_size in (50, 101):
        return [3, 4, 23, 3]
    elif resnet_size == 152:
        return [3, 8, 36, 3]
    raise ValueError("Unsuported resnet size: ", resnet_size)


def get_block_type(resnet_size):
    if resnet_size in (18, 34):
        return BasicBlock
    elif resnet_size in (50, 101, 152):
        return Bottleneck
    raise ValueError("Unsuported resnet size ", resnet_size)


class ResidualBase(nn.Module):
    def __init__(self):
        super().__init__()
        self.cur_in_ch = 64

    def make_layer(self, block, out_ch, size, stride=1, shortcut="downsample"):
        layers = []
        stride_mean = stride
        if isinstance(stride, tuple):
            stride_mean = sum(stride) / len(stride)

        if shortcut == "upsample" and (stride_mean > 1
                                       or self.cur_in_ch != out_ch * block.expansion):
            shortcut = nn.ConvTranspose2d(self.cur_in_ch, out_ch * block.expansion,
                                          kernel_size=3, stride=stride,
                                          padding=get_padding(3),
                                          output_padding=get_padding(stride), bias=False)
            layers.append(block(self.cur_in_ch, out_ch, stride=stride,
                                mid_ch=self.cur_in_ch // block.expansion,
                                upsample=shortcut))
        elif shortcut == "downsample" and (stride_mean > 1
                                           or self.cur_in_ch != out_ch * block.expansion):
            shortcut = nn.Sequential(
                nn.Conv2d(self.cur_in_ch, out_ch * block.expansion, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_ch * block.expansion),
            )
            layers.append(block(self.cur_in_ch, out_ch, stride, downsample=shortcut))
        else:
            layers.append(block(self.cur_in_ch, out_ch))
        self.cur_in_ch = out_ch * block.expansion

        for _ in range(1, size):
            layers.append(block(self.cur_in_ch, out_ch))
        return nn.Sequential(*layers)


class ResidualEncoder(ResidualBase):
    def __init__(self, opts=None):
        super().__init__()
        if opts is None:
            opts = {"input_channels": 1, "conv_kernel_size": 7,
                    "max_pool": 1, "resnet_size": 18}
        self._opts = opts
        self.cur_in_ch = 64
        self.block_sizes = get_block_sizes(opts["resnet_size"])
        self.block_type = get_block_type(opts["resnet_size"])

        self.conv1 = nn.Conv2d(opts["input_channels"], out_channels=64,
                               kernel_size=opts["conv_kernel_size"], stride=(2, 2),
                               padding=get_padding(opts["conv_kernel_size"]), bias=False)
        self.bn1 = nn.BatchNorm2d(self.cur_in_ch)
        self.relu1 = nn.ReLU(inplace=True)
        if opts["max_pool"] == 1:
            self.max_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=get_padding(3))
            stride1 = (1, 1)
        elif opts["max_pool"] == 0:
            self.max_pool = None
            stride1 = (2, 2)
        elif opts["max_pool"] == 2:
            self.max_pool = None
            stride1 = (1, 1)
        else:
            raise ValueError("Unknown max_pool option")

        self.layer1 = self.make_layer(self.block_type, 64, self.block_sizes[0], stride1)
        self.layer2 = self.make_layer(self.block_type, 128, self.block_sizes[1], (2, 2))
        self.layer3 = self.make_layer(self.block_type, 256, self.block_sizes[2], (2, 2))
        self.layer4 = self.make_layer(self.block_type, 512, self.block_sizes[3], (2, 2))

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        if self.max_pool is not None:
            x = self.max_pool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

    def model_opts(self):
        return self._opts


class Classifier(nn.Module):
    def __init__(self, opts=None):
        super().__init__()
        if opts is None:
            opts = {"input_channels": 512, "pooling": "avg", "num_classes": 2}
        self._opts = opts
        self._layer_output = dict()
        if opts["pooling"] == "avg":
            self.pooling = lambda x: torch.mean(x, dim=-1)
        elif opts["pooling"] == "max":
            self.pooling = lambda x: torch.max(x, dim=-1)[0]
        else:
            raise ValueError("Unkown pooling option")
        self.linear = nn.Linear(opts["input_channels"], opts["num_classes"])

    def forward(self, x):
        x = x.view(x.size(0), x.size(1), -1)
        hidden_layer = self.pooling(x)
        hidden_layer = hidden_layer.view(hidden_layer.size(0), -1)
        self._layer_output["hidden_layer_1"] = hidden_layer
        output_layer = self.linear(hidden_layer)
        self._layer_output["output_layer"] = output_layer
        return output_layer

    def model_opts(self):
        return self._opts

def _assemble(encoder, classifier):
    model = nn.Sequential(OrderedDict([('encoder', encoder), ('classifier', classifier)]))
    model.eval()
    return model


def build_model(encoderOpts, num_classes, device):
    """Build a fresh model (random init) with a num_classes-wide head."""
    classifierOpts = {'input_channels': 512, 'pooling': 'avg', 'num_classes': num_classes}
    encoder = ResidualEncoder(encoderOpts)
    classifier = Classifier(classifierOpts)
    return _assemble(encoder, classifier), encoder, classifier, classifierOpts


def load_model_from_pk(path, device):
    """Load a .pk file. Returns (model, classes_dict, dataOpts)."""
    obj = torch.load(path, map_location=device, weights_only=False)
    encoder = ResidualEncoder(obj['encoderOpts'])
    encoder.load_state_dict(obj['encoderState'])
    classifier = Classifier(obj['classifierOpts'])
    classifier.load_state_dict(obj['classifierState'])
    return _assemble(encoder, classifier), obj.get('classes', {}), obj.get('dataOpts', {})


# .checkpoint files (Data/model_output/*) store no opts, so these are supplied
# from that training run's config. Verify against your own TRAIN.log if in doubt.
CHECKPOINT_ENCODER_OPTS = {'input_channels': 1, 'conv_kernel_size': 7,
                           'max_pool': 2, 'resnet_size': 18}
CHECKPOINT_DATA_OPTS = {'sr': 384000, 'preemphases': 0.98, 'n_fft': 256,
                        'hop_length': 128, 'n_freq_bins': 256, 'fmin': 18000,
                        'fmax': 90000, 'freq_compression': 'linear',
                        'min_level_db': -100, 'ref_level_db': 20}


def load_any_model(path, device):
    """Load either a .pk or a .checkpoint file."""
    obj = torch.load(path, map_location=device, weights_only=False)
    if 'encoderOpts' in obj:
        return load_model_from_pk(path, device)
    if 'modelState' in obj:
        classes = obj.get('classes', {})
        classifierOpts = {'input_channels': 512, 'pooling': 'avg',
                          'num_classes': len(classes)}
        encoder = ResidualEncoder(CHECKPOINT_ENCODER_OPTS)
        classifier = Classifier(classifierOpts)
        ms = obj['modelState']
        enc_state = {k[len('encoder.'):]: v for k, v in ms.items() if k.startswith('encoder.')}
        cls_state = {k[len('classifier.'):]: v for k, v in ms.items() if k.startswith('classifier.')}
        encoder.load_state_dict(enc_state)
        classifier.load_state_dict(cls_state)
        return _assemble(encoder, classifier), classes, dict(CHECKPOINT_DATA_OPTS)
    raise ValueError(f'Unknown model format: {list(obj.keys())}')


print('Architecture + loading functions defined')


In [ ]:
# Cell 6: Transforms & Dataset
#
# WHY THIS CELL CHANGED (see AGENTS.md "Windowing fix"):
# The clips are 15 ms - 2.6 s long (median ~375 ms) but every model sees a
# 20 ms window. The previous version kept spec[:seq_len] -- the FIRST 20 ms of
# every clip, always the same one -- so ~95 % of each clip was never used and
# the network memorised 674 fixed inputs. Random 20 ms crops are what the
# official ANIMAL-SPOT code trains on; this cell implements a deliberately
# STRICTER variant of that (crops restricted to the loudest 20 % of windows,
# to avoid sampling the silence between pulses), and aggregates the top-5
# loudest windows at prediction time.
#   Measured effect of the windowing change alone: classifier 0.754 -> 0.853,
#   detector m09 0.866 -> 0.933 (AGENTS.md 8.5). It multiplies distinct training
#   inputs by ~x56 (classifier) / ~x43 (detector).
#   WINDOW_MODE = 'first' reproduces the old *window choice* (not the old front end).
#
# Front end = official animal_spot/data/transforms.py, step for step:
#   mono mean -> resample (kaiser_best) -> pre-emphasis 0.98 ->
#   torch.stft(center=False) / sqrt(sum(w^2)) -> power -> keep fmin..fmax bins
#   -> NEAREST interpolation to n_freq_bins -> 10*log10 floored at -100 dB
#   -> per-window min-max normalisation (the official models were trained with
#      min_max_norm=true, and predict.py normalises each window on its own).
# The old front end used scipy.ndimage.zoom (cubic spline) on the *power*
# spectrum, which rings negative and is clipped to the -100 dB floor.
# Verified bit-exact against the vendored official code (max|diff| = 0.000e+00).

import random as _random
import hashlib
import inspect
import json as _json


def get_class_from_filename(filename):
    """Extract class label from BatSpot filename (before first '-')."""
    return os.path.basename(filename).split('-', 1)[0]


def load_audio_file(path, target_sr):
    """Load audio (mono mean, like the official loader) and resample to target_sr."""
    audio, native_sr = sf.read(path, dtype='float32')
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if native_sr != target_sr:
        audio = resampy.resample(audio, native_sr, target_sr, filter='kaiser_best')
    return audio.astype(np.float32), target_sr


def clip_to_db_spectrogram(audio, cfg, preemphasis=0.98, min_level_db=-100):
    """Official front end up to (and incl.) amplitude->dB. Returns (frames, n_freq_bins) float32.
    NOT normalised: min-max is applied per 20 ms window later."""
    y = torch.from_numpy(audio)[None]
    y = torch.cat((y[:, :1], y[:, 1:] - preemphasis * y[:, :-1]), dim=-1)
    win = torch.hann_window(cfg['n_fft'])
    S = torch.stft(y, n_fft=cfg['n_fft'], hop_length=cfg['hop_length'], window=win,
                   center=False, onesided=True, return_complex=True)
    S = torch.view_as_real(S).transpose(1, 2)            # (1, frames, bins, 2)
    S = S / win.pow(2).sum().sqrt()
    S = S.pow(2).sum(-1)                                  # power, (1, frames, n_fft/2+1)
    n_fft = (S.size(2) - 1) * 2
    lo = int(max(0, np.floor(n_fft * cfg['fmin'] / cfg['sr'])))
    hi = int(min(n_fft - 1, np.ceil(n_fft * cfg['fmax'] / cfg['sr'])))
    S = S[:, :, lo:hi]
    S = torch.nn.functional.interpolate(S[None], size=(S.size(1), cfg['n_freq_bins']),
                                        mode='nearest')[0]
    S = torch.clamp(S, min=float(np.exp(min_level_db / 10 * np.log(10))))
    return (10 * torch.log10(S))[0].numpy().astype(np.float32)


def minmax_normalize(spec):
    """Min-max normalise to [0, 1] (official MinMaxNormalize)."""
    spec = spec - spec.min()
    m = spec.max()
    return spec / m if m > 0 else spec


def pad_window(win, seq_len):
    """Centre-pad a short window with zeros to seq_len frames.

    NOTE: the official PaddedSubsequenceSampler pads at a RANDOM offset when
    random=True (training) and at the centre otherwise; this cell always centre-pads.
    Only reachable for clips shorter than one 20 ms window -- 0 of 964 in this
    dataset -- so the deviation is documented rather than randomised.
    """
    n = win.shape[0]
    if n >= seq_len:
        return win
    out = np.zeros((seq_len, win.shape[1]), dtype=np.float32)
    p = (seq_len - n) // 2
    out[p:p + n] = win
    return out


def window_scores(db, seq_len, stride):
    """Candidate window starts + a loudness score (mean of the per-frame peak dB)."""
    n = db.shape[0]
    if n <= seq_len:
        return np.array([0]), np.array([0.0])
    peak = db.max(axis=1)
    cs = np.concatenate([[0.0], np.cumsum(peak)])
    starts = np.arange(0, n - seq_len + 1, stride)
    return starts, (cs[starts + seq_len] - cs[starts]) / seq_len


def train_window_starts(starts, scores, mode, top_frac):
    """Windows a training clip may be cropped at: the loudest `top_frac` (>= 3).

    FIX (Space Bunny Free): this used to be a percentile threshold,
        starts[scores >= np.percentile(scores, 100 * (1 - top_frac))]
    Ties make a threshold select EVERYTHING. Measured on this dataset: kept-fraction
    median 0.202 but max 1.000, with 11/964 clips above 0.30 and 3 clips fully
    degenerate -- those 3 got uniform crops over the whole clip, silence included,
    which is label noise. Now it takes exactly ceil(top_frac * n) windows by score
    (stable sort, so ties break by position), which is deterministic.
    """
    if mode == 'first':
        return starts[:1]
    n = len(starts)
    k = min(max(3, int(np.ceil(n * top_frac))), n)
    order = np.argsort(-scores, kind='stable')
    return np.sort(starts[order[:k]])


def eval_window_starts(starts, scores, mode, k, seq_len):
    """Top-k loudest windows, greedy NMS so consecutive picks overlap by < half a window.

    NOTE: 'non-overlapping' would be wrong -- the guarantee is >= seq_len/2 frames apart
    (10 ms at 192 kHz), which happens to equal the official detector's 10 ms prediction hop.
    """
    if mode == 'first':
        return starts[:1]
    chosen = []
    for i in np.argsort(-scores, kind='stable'):
        if all(abs(starts[i] - starts[j]) >= seq_len // 2 for j in chosen):
            chosen.append(i)
        if len(chosen) >= k:
            break
    return starts[chosen]


def _augment_spec(spec):
    """Optional on-the-fly spectrogram augmentation (training only, USE_AUGMENTATION).
    Input shape (seq_len, n_freq_bins). Time shift, Gaussian noise, SpecAugment masks.
    NOTE: random window cropping already provides most of the useful augmentation;
    this stays OFF on the main path and is A/B tested in the ablation cell."""
    seq_len, n_freq = spec.shape
    spec = spec.copy()
    shift = int(np.random.uniform(-0.20, 0.20) * seq_len)
    if shift > 0:
        spec = np.concatenate([np.zeros((shift, n_freq)), spec[:-shift]], axis=0)
    elif shift < 0:
        spec = np.concatenate([spec[-shift:], np.zeros((-shift, n_freq))], axis=0)
    spec = np.clip(spec + np.random.normal(0, 0.01, spec.shape), 0.0, 1.0)
    f_mask = int(np.random.uniform(0, 0.20) * n_freq)
    if f_mask > 0:
        f0 = np.random.randint(0, max(1, n_freq - f_mask))
        spec[:, f0:f0 + f_mask] = 0.0
    t_mask = int(np.random.uniform(0, 0.20) * seq_len)
    if t_mask > 0:
        t0 = np.random.randint(0, max(1, seq_len - t_mask))
        spec[t0:t0 + t_mask, :] = 0.0
    return spec.astype(np.float32)


class WindowedBatDataset(Dataset):
    """Clip-level dataset served as 20 ms windows.

    train=True : __getitem__(i) -> (window, label); a RANDOM window from the
                 loudest TRAIN_TOP_FRAC of clip i (new crop every epoch).
    train=False: flat list of the TEST_TOPK loudest windows of every clip;
                 __getitem__ -> (window, label, clip_index).
                 Use predict_proba() to average the windows back to clip level.

    The full-clip dB spectrogram is cached once to disk (np.save) and memory-
    mapped, so a window read costs microseconds.
    """
    _CACHE_ROOT = os.path.join(WORKING_DIR, 'spec_cache')

    def _cache_version():
        """Cache tag derived from the front-end source.

        Was a hand-maintained string ('v2'), so editing clip_to_db_spectrogram without bumping
        it silently reused stale spectrograms. Deriving it from the function source removes that
        failure mode. inspect.getsource needs the source to be locatable on disk, which is true
        for Jupyter/nbconvert/papermill but not for code exec'd from a string -- hence the
        fallback rather than letting dataset construction die.
        """
        try:
            src = inspect.getsource(clip_to_db_spectrogram)
            return 'v2-' + hashlib.md5(src.encode()).hexdigest()[:6]
        except (OSError, TypeError):
            return 'v2-fallback(no-source)'

    _CACHE_VERSION = _cache_version()

    def __init__(self, file_names, class_to_idx, cfg, train=False, augment=False,
                 mode=None, topk=None):
        self.file_names = list(file_names)
        self.class_to_idx = class_to_idx
        self.cfg = cfg
        self.train = train
        self.augment = augment and train
        self.mode = WINDOW_MODE if mode is None else mode
        if self.mode not in ('energy_crop', 'first'):
            raise ValueError(f"WINDOW_MODE must be 'energy_crop' or 'first', got {self.mode!r}")
        self.topk = TEST_TOPK if topk is None else int(topk)
        self.seq_len = int(float(cfg['sequence_len']) / 1000 * cfg['sr'] / cfg['hop_length'])
        key = (f"{self._CACHE_VERSION}_{cfg['sr']}_{cfg['n_fft']}_{cfg['hop_length']}_"
               f"{cfg['n_freq_bins']}_{cfg['fmin']}_{cfg['fmax']}")
        self.cache_dir = os.path.join(self._CACHE_ROOT, hashlib.md5(key.encode()).hexdigest()[:8])
        # FIX: the cache file name was the bare basename, so two different DATA_DIRs with
        # colliding basenames shared one cache entry and one silently overwrote the other.
        # The source directory is part of the name now.
        self._dir_tag = hashlib.md5(
            os.path.dirname(os.path.abspath(self.file_names[0])).encode()).hexdigest()[:6]
        os.makedirs(self.cache_dir, exist_ok=True)
        self._build_cache()

        # FIX: a clip whose wav is unreadable/too short used to raise out of torch.stft and
        # kill the whole run. Failures are now reported and the clips dropped, visibly.
        usable = [f for f in self.file_names if self._cache_is_valid(f)]
        if len(usable) != len(self.file_names):
            print(f'[WindowedBatDataset] WARNING: dropped {len(self.file_names) - len(usable)} '
                  f'unusable clip(s) of {len(self.file_names)}')
        self.file_names = usable
        if not self.file_names:
            raise RuntimeError(f'no usable clips under {cfg.get("_src", "?")}')

        self.labels = np.array([class_to_idx[get_class_from_filename(f)] for f in self.file_names])
        # window bookkeeping
        self.train_starts, self.items = [], []
        n_short = 0
        for i, f in enumerate(self.file_names):
            db = np.load(self._spec_path(f), mmap_mode='r')
            starts, scores = window_scores(db, self.seq_len, WINDOW_STRIDE)
            if db.shape[0] <= self.seq_len:
                n_short += 1
            if train:
                self.train_starts.append(train_window_starts(starts, scores, self.mode, TRAIN_TOP_FRAC))
            else:
                self.items += [(i, int(s)) for s in
                               eval_window_starts(starts, scores, self.mode, self.topk, self.seq_len)]
        if n_short:
            # Informational, not an error: the pad path below handles these correctly.
            print(f'[WindowedBatDataset] note: {n_short}/{len(self.file_names)} clip(s) are shorter '
                  f'than one {self.seq_len}-frame window; they are zero-padded AFTER min-max.')

    def _spec_path(self, f):
        return os.path.join(self.cache_dir,
                            f'{self._dir_tag}_{os.path.basename(f)}.npy')

    def _meta_path(self, f):
        return self._spec_path(f) + '.meta.json'

    def _cache_is_valid(self, f):
        """FIX: the old check was `os.path.exists(...)`, so re-running clip extraction with
        OVERWRITE_EXISTING_CLIPS=True, or pointing DATA_DIR at a re-extracted copy, silently
        trained on the OLD spectrograms. Now the source wav's size+mtime are recorded alongside
        the cache entry and validated on load."""
        p, m = self._spec_path(f), self._meta_path(f)
        if not (os.path.exists(p) and os.path.exists(m)):
            return False
        try:
            with open(m) as fh:
                rec = _json.load(fh)
            st = os.stat(f)
        except Exception:
            return False
        return (rec.get('version') == self._CACHE_VERSION
                and rec.get('size') == st.st_size
                and rec.get('mtime_ns') == st.st_mtime_ns)

    def _build_cache(self):
        missing = [f for f in self.file_names if not self._cache_is_valid(f)]
        if not missing:
            return
        print(f'[WindowedBatDataset] caching {len(missing)} clips -> {self.cache_dir}')
        failed = []
        for f in tqdm(missing, desc='cache', leave=False):
            try:
                audio, _ = load_audio_file(f, self.cfg['sr'])
                spec = clip_to_db_spectrogram(audio, self.cfg)
                # FIX: an all-NaN wav produced a spectrogram that was fed straight to the model.
                spec = np.nan_to_num(spec, nan=-100.0, posinf=0.0, neginf=-100.0)
                # FIX: np.save is not atomic, so an interrupted build left a truncated .npy that
                # every later run happily accepted and mmap'd as garbage.
                tmp = self._spec_path(f) + '.tmp.npy'
                np.save(tmp, spec)
                os.replace(tmp, self._spec_path(f))
                st = os.stat(f)
                with open(self._meta_path(f), 'w') as fh:
                    _json.dump({'version': self._CACHE_VERSION, 'size': st.st_size,
                                'mtime_ns': st.st_mtime_ns, 'frames': int(spec.shape[0])}, fh)
            except Exception as e:                      # noqa: BLE001 - report, do not abort
                failed.append((os.path.basename(f), repr(e)))
        if failed:
            print(f'[WindowedBatDataset] WARNING: {len(failed)} clip(s) failed and will be SKIPPED:')
            for nm, err in failed[:10]:
                print(f'    {nm}: {err}')
            if len(failed) > 10:
                print(f'    ... and {len(failed) - 10} more')

    def _mmap(self, i):
        """Per-instance mmap cache: __getitem__ runs ~5k times/epoch and re-opening the .npy
        each time is pure overhead. Created lazily, so each DataLoader worker gets its own."""
        c = getattr(self, '_mmap_cache', None)
        if c is None:
            c = self._mmap_cache = {}
        if i not in c:
            if len(c) > 128:
                c.clear()
            c[i] = np.load(self._spec_path(self.file_names[i]), mmap_mode='r')
        return c[i]

    def _window(self, i, start):
        db = self._mmap(i)
        win = np.array(db[start:start + self.seq_len]).astype(np.float32)
        # FIX (Space Bunny Free) -- ORDER OF OPERATIONS. This used to be
        #     minmax_normalize(pad_window(win, self.seq_len))
        # which inserts a literal 0.0 into a dB array BEFORE normalising. Real dB values are
        # <= ~0 dB, so that 0.0 is normally the array MAXIMUM: min-max then divides by the
        # padding and maps every padded row to 1.0, i.e. silence handed to the model as the
        # loudest thing in the window. Measured on a 12 ms clip inside a 20 ms window:
        # padded rows had mean 1.0000 here vs 0.0000 with the official order.
        # The official pipeline normalises first and pads after
        # (animal_spot/data/audiodataset.py: t_norm then t_subseq), so the pad region is 0.0.
        win = minmax_normalize(win)
        return pad_window(win, self.seq_len).astype(np.float32)

    def __len__(self):
        return len(self.file_names) if self.train else len(self.items)

    def __getitem__(self, idx):
        if self.train:
            win = self._window(idx, int(_random.choice(self.train_starts[idx])))
            if self.augment:
                win = _augment_spec(win)
            return torch.from_numpy(win).unsqueeze(0), int(self.labels[idx])
        i, start = self.items[idx]
        return torch.from_numpy(self._window(i, start)).unsqueeze(0), int(self.labels[i]), i


def predict_proba(model, dataset, device, batch_size=256):
    """Clip-level probabilities: softmax averaged over each clip's windows.
    `dataset` must be a train=False WindowedBatDataset. Returns (probs[n_clips, C], labels[n_clips])."""
    assert not dataset.train, 'predict_proba needs an evaluation (train=False) dataset'
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    sums, counts = None, np.zeros(len(dataset.file_names))
    with torch.no_grad():
        for inputs, _, clip_idx in loader:
            p = torch.softmax(model(inputs.to(device)).float(), dim=1).cpu().numpy()
            if sums is None:
                sums = np.zeros((len(dataset.file_names), p.shape[1]))
            for row, ci in zip(p, clip_idx.numpy()):
                sums[ci] += row
                counts[ci] += 1
    # FIX: a clip with zero windows used to yield an all-zero probability row whose argmax is
    # class 0 -- a silent free "correct" prediction for noise / classifier class 0. Not
    # reachable on this dataset (min 1 window per clip) but it must never be silent.
    if not (counts > 0).all():
        _bad = [dataset.file_names[i] for i in np.flatnonzero(counts == 0)][:5]
        raise RuntimeError(f'{int((counts == 0).sum())} clip(s) produced no windows, e.g. {_bad}')
    return sums / counts[:, None], dataset.labels.copy()


print('Windowed transforms + WindowedBatDataset defined '
      f'(WINDOW_MODE={WINDOW_MODE!r}, train top {TRAIN_TOP_FRAC:.0%} loudest windows, test top-{TEST_TOPK})')
print(f'[cache] version tag {WindowedBatDataset._CACHE_VERSION} (derived from the front-end source)')


In [ ]:
# Cell 7: Data Discovery

import re as _re

# Find all wav files
all_wavs = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.wav'), recursive=True))
if not all_wavs:
    # Try flat structure
    all_wavs = sorted(glob.glob(os.path.join(DATA_DIR, '*.wav')))

print(f'Found {len(all_wavs)} wav files')

# Auto-detect classes from folder names
class_counts = Counter()
for w in all_wavs:
    cls = get_class_from_filename(w)
    class_counts[cls] += 1

all_classes = sorted(class_counts.keys())
print(f'Classes ({len(all_classes)}):')
for c in all_classes:
    print(f'  {c}: {class_counts[c]} files')

# Update classifier config with actual num_classes
CLS_CONFIG['num_classes'] = len(all_classes)
print(f'\nClassifier num_classes: {CLS_CONFIG["num_classes"]}')

# Detector class mapping: noise -> 0, all bat species -> 1 (call)
DET_CLASSES = ['noise', 'call']

# Classifier class mapping: sorted species classes (incl. noise)
CLS_CLASSES = all_classes
CLS_CLASS_TO_IDX = {c: i for i, c in enumerate(all_classes)}
CLS_IDX_TO_CLASS = {i: c for c, i in CLS_CLASS_TO_IDX.items()}

print(f'\nDetector classes (output labels): {DET_CLASSES}')
print(f'Classifier mapping: {CLS_CLASS_TO_IDX}')

# ---------------------------------------------------------------------------
# DATA SPLIT: 70/15/15 train/val/test, stratified by species (incl. noise).
# Stratifying by species preserves the 70/15/15 binary (noise vs call) proportions,
# and ONE shared split is used by detector and classifier so the cascade has no leakage.
#
# RECORDING LEAKAGE WARNING: the filename carries the tape (recording) id. This dataset has
# only ~28 recordings (heti is a single one), so a random clip-level split puts clips of the
# SAME recording in train and test. The network can then score well by recognising the
# recording (background, mic distance) rather than the species -> optimistic numbers.
# The checks below quantify it, and SPLIT_BY_RECORDING = True (Cell 2) holds whole
# recordings out instead. The grouped number is produced by the last cell.
# ---------------------------------------------------------------------------

# FIX (Space Bunny Free): this used to be
#     parts = basename.split('_');  return parts[3] if len(parts) >= 6 else basename
# i.e. a POSITIONAL index into the 6-field CLASS-LABEL_ID_YEAR_TAPE_START_END.wav layout,
# falling back to the whole file name. Cell 8 -- enabled by default, RUN_CLIP_EXTRACTION=True
# -- writes different names, so every clip produced by this notebook's OWN extractor counted as
# its own recording. Two things then broke quietly:
#   * the leakage check below printed a false all-clear (0/N shared recordings), and
#   * SPLIT_BY_RECORDING = True became a plain random split while claiming to hold out
#     recordings.
# Both layouts carry the Raven session stamp YYYYMMDD + HHMMSS, which identifies the recording,
# so match on THAT rather than on a position.
#
# The separator differs between the layouts and must not be assumed:
#   Raven/manual : acsh-bat_3379376_2026_20260525-192000_91577_92135.wav       -> 20260525-192000
#   Cell 8 output: acsh-bat_acsh_devon_acsh_devon_20260521_204000_1234_5678.wav -> 20260521-204000
# so the pattern accepts EITHER '-' or '_', and is searched ANYWHERE in the stem rather than as a
# whole underscore-delimited field (Cell 8 splits the date and the time into two fields).
# A first version of this fix used '^\d{8}-\d{6}$' and therefore did NOT match Cell 8's real
# output on this repo -- it silently fell back to per-clip groups, exactly as the old code did.
# The n_groups == n_files warning below is what caught it; the pattern is now separator-agnostic.
#
# Keying on the session stamp alone is correct: clips of DIFFERENT species recorded in one session
# genuinely share a tape (acsh/alte/noise clips all carry 20260521-192000).
_TS_RE = _re.compile(r'(\d{8})[-_](\d{6})')


def recording_id(path):
    """Tape (recording) id from a BatSpot clip filename; stem as a last resort."""
    stem = os.path.basename(path)[:-4]
    m = _TS_RE.search(stem)
    return f'{m.group(1)}-{m.group(2)}' if m else stem


all_rec_groups = {recording_id(w) for w in all_wavs}
all_species_labels = [get_class_from_filename(w) for w in all_wavs]

# Recording count PER CLASS -- the number that actually predicts which classes will look
# perfect in an in-split evaluation. A class recorded once cannot be held out at all.
_rec_per_class = {}
for _w, _s in zip(all_wavs, all_species_labels):
    _rec_per_class.setdefault(_s, set()).add(recording_id(_w))
print(f'\nRecordings: {len(all_rec_groups)} distinct tape ids for {len(all_wavs)} clips')
print('  recordings per class (a class with 1 cannot be validated on a new recording):')
for _c in all_classes:
    _n = len(_rec_per_class.get(_c, ()))
    _flag = '   <-- single recording' if _n <= 1 else ('   <-- 2 recordings' if _n == 2 else '')
    print(f'    {_c:<6} {_n:>3} recording(s){_flag}')
if len(all_rec_groups) >= len(all_wavs):
    print('  WARNING: every clip is its own group. recording_id() could not find a tape id in '
          'these filenames, so the leakage check below is meaningless and SPLIT_BY_RECORDING '
          'will silently behave like a random split.')

if SPLIT_BY_RECORDING:
    from sklearn.model_selection import StratifiedGroupKFold
    _folds = [te for _, te in StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=1)
              .split(all_wavs, all_species_labels, [recording_id(w) for w in all_wavs])]
    test_wavs = [all_wavs[i] for i in _folds[0]]
    val_wavs = [all_wavs[i] for i in _folds[1]]
    _held = set(_folds[0]) | set(_folds[1])
    train_wavs = [w for i, w in enumerate(all_wavs) if i not in _held]
    train_labels = [get_class_from_filename(w) for w in train_wavs]
    val_labels = [get_class_from_filename(w) for w in val_wavs]
    test_labels = [get_class_from_filename(w) for w in test_wavs]
    print('SPLIT_BY_RECORDING = True: whole recordings are held out. NOTE this is NOT 70/15/15 '
          '(7 folds -> ~28.6 % held out) and classes with few recordings end up in one split '
          'only -- read per-class numbers with care.')
else:
    train_wavs, test_wavs, train_labels, test_labels = train_test_split(
        all_wavs, all_species_labels, test_size=0.15, random_state=42,
        stratify=all_species_labels)
    train_wavs, val_wavs, train_labels, val_labels = train_test_split(
        train_wavs, train_labels, test_size=0.15/0.85, random_state=42,
        stratify=train_labels)

print(f'\nSplit: train={len(train_wavs)}, val={len(val_wavs)}, test={len(test_wavs)}')
print('Train class dist:', Counter(train_labels))
print('Val class dist:  ', Counter(val_labels))
print('Test class dist: ', Counter(test_labels))

_train_rec = {recording_id(w) for w in train_wavs}
_n_leak = sum(recording_id(w) in _train_rec for w in test_wavs)
print(f'\nLEAKAGE CHECK: {_n_leak}/{len(test_wavs)} test clips come from a recording that also '
      f'appears in train ({len(all_rec_groups)} recordings in total).')
if _n_leak:
    print('  -> test scores are optimistic about NEW recordings; set SPLIT_BY_RECORDING = True '
          'for a stricter (but noisier) estimate, or read the grouped number in the last cell.')

# ---------------------------------------------------------------------------
# CLASS PRIORS -- needed by the prior-corrected detector gate in Cell 14.
#
# WHY: make_weighted_sampler() draws every class with probability exactly 1/n_classes, so the
# detector is trained under a 50/50 noise/call prior while val/test is ~20/80. Probabilities
# produced under the training prior are therefore shifted towards 'call' by
#     (prior_call/0.5) / (prior_noise/0.5)
# in odds -- about 4.2x at these priors. That is why the Kaggle run showed m09 with noise
# PRECISION 0.70 against recall 0.93, and why the val-tuned gate thresholds had to sink to
# 0.30 / 0.40 / 0.10 to compensate. A threshold fitted that way encodes the 20 % noise prior
# and will not transfer to real recordings, where noise is >95 % of windows.
#
# The classifier's own argmax is deliberately NOT prior-corrected: its errors point the other
# way (10 of 21 errors are false 'acsh', the MOST common class), so prior shift is not its
# problem -- its problem is acoustic confusion between acsh and alte.
# ---------------------------------------------------------------------------
def _empirical_prior(labels, n):
    c = Counter(labels)
    t = max(len(labels), 1)
    return np.array([c.get(i, 0) / t for i in range(n)])


_wav_index = {w: i for i, w in enumerate(all_wavs)}
_is_noise = [s == 'noise' for s in all_species_labels]
# Detector OUTPUT index 0 = noise, 1 = call. _is_noise is True for noise, so it must be INVERTED
# here -- an earlier version of this cell passed int(_is_noise) straight in, which swapped the
# two priors and would have made the prior correction in Cell 14 anti-correct.
DET_TRAIN_PRIOR = _empirical_prior([0 if _is_noise[_wav_index[w]] else 1 for w in train_wavs], 2)
DET_VAL_PRIOR = _empirical_prior([0 if _is_noise[_wav_index[w]] else 1 for w in val_wavs], 2)
DET_EVAL_PRIOR = _empirical_prior([0 if _is_noise[_wav_index[w]] else 1 for w in test_wavs], 2)
CLS_TRAIN_PRIOR = _empirical_prior([CLS_CLASS_TO_IDX[s] for s in train_labels], len(all_classes))
CLS_EVAL_PRIOR = _empirical_prior([CLS_CLASS_TO_IDX[s] for s in test_labels], len(all_classes))
# The sampler overrides both with uniform.
DET_EFFECTIVE_TRAIN_PRIOR = np.full(2, 0.5)
CLS_EFFECTIVE_TRAIN_PRIOR = np.full(len(all_classes), 1.0 / len(all_classes))

print('\nCLASS PRIORS')
print(f'  detector  train (effective, sampler) noise={DET_EFFECTIVE_TRAIN_PRIOR[0]:.3f} '
      f'call={DET_EFFECTIVE_TRAIN_PRIOR[1]:.3f}   test noise={DET_EVAL_PRIOR[0]:.3f} '
      f'call={DET_EVAL_PRIOR[1]:.3f}')
print('    -> prior-correction factor (test/train): '
      f'noise x{(DET_EVAL_PRIOR[0] / DET_EFFECTIVE_TRAIN_PRIOR[0]):.2f}  '
      f'call x{(DET_EVAL_PRIOR[1] / DET_EFFECTIVE_TRAIN_PRIOR[1]):.2f}')
print('  classifier train (effective, sampler) uniform 1/%d; test:' % len(all_classes))
print('    ' + '  '.join(f'{c}={CLS_EVAL_PRIOR[i]:.3f}' for i, c in enumerate(CLS_CLASSES)))


In [ ]:
# Cell 8: Clip Extraction (Python port of export_clips.R)

def extract_clips_from_selections(raw_audio_dir, selections_dir, output_dir, target_sr=384000):
    """
    Extract audio clips from raw recordings using Raven selection tables.
    Python port of Data/scripts/export_clips.R.
    """
    os.makedirs(output_dir, exist_ok=True)
    extracted = 0

    # Iterate over species folders in selections_dir
    for species in sorted(os.listdir(selections_dir)):
        species_dir = os.path.join(selections_dir, species)
        if not os.path.isdir(species_dir):
            continue

        for sel_file in sorted(glob.glob(os.path.join(species_dir, '*.txt'))):
            # Read Raven selection table (tab-separated)
            with open(sel_file, 'r') as f:
                lines = f.readlines()
            if len(lines) < 2:
                continue

            # Parse header
            header = lines[0].strip().split('\t')
            try:
                begin_idx = header.index('Begin Time (s)')
                end_idx = header.index('End Time (s)')
                view_idx = header.index('View')
            except ValueError:
                continue

            # Find matching raw audio file by date/time in selection table filename
            # Selection filename format: YYYYMMDD_HHMMSS_species.txt
            sel_name = os.path.basename(sel_file).replace('.txt', '')
            date_time = '_'.join(sel_name.split('_')[:2])
            matching_audio = None
            for audio_file in glob.glob(os.path.join(raw_audio_dir, '*.WAV')):
                audio_name = os.path.basename(audio_file).replace('.WAV', '')
                if date_time in audio_name:
                    matching_audio = audio_file
                    break
            if matching_audio is None:
                continue

            # Load audio
            audio, native_sr = sf.read(matching_audio, dtype='float32')
            if audio.ndim > 1:
                audio = audio[:, 0]

            # Extract each selection
            for line in lines[1:]:
                parts = line.strip().split('\t')
                if len(parts) <= max(begin_idx, end_idx, view_idx):
                    continue
                view = parts[view_idx]
                if 'Spectrogram' not in view:
                    continue
                begin_time = float(parts[begin_idx])
                end_time = float(parts[end_idx])
                start_sample = int(begin_time * native_sr)
                end_sample = int(end_time * native_sr)
                clip = audio[start_sample:end_sample]
                if len(clip) == 0:
                    continue
                start_ms = int(begin_time * 1000)
                end_ms = int(end_time * 1000)
                out_name = f'{species}-bat_{date_time}_{sel_name}_{start_ms}_{end_ms}.wav'
                out_path = os.path.join(output_dir, out_name)
                sf.write(out_path, clip, native_sr)
                extracted += 1

    print(f'Extracted {extracted} clips to {output_dir}')
    return extracted

# Extraction is opt-in, and refuses to clobber an existing dataset.
if not RUN_CLIP_EXTRACTION:
    print('Clip extraction SKIPPED (RUN_CLIP_EXTRACTION = False in Cell 2).')
    print(f'Using the clips already present in: {DATA_DIR}')
elif not (RAW_AUDIO_DIR and SELECTIONS_DIR):
    print('Clip extraction skipped: set RAW_AUDIO_DIR and SELECTIONS_DIR in Cell 2.')
else:
    _existing = glob.glob(os.path.join(DATA_DIR, '**', '*.wav'), recursive=True)
    if _existing and not OVERWRITE_EXISTING_CLIPS:
        print(f'Clip extraction ABORTED: {len(_existing)} wav files already exist in')
        print(f'  {DATA_DIR}')
        print('Re-extracting would overwrite them. Set OVERWRITE_EXISTING_CLIPS = True')
        print('in Cell 2 to force it, or point DATA_DIR at an empty folder.')
    else:
        if _existing:
            print(f'WARNING: overwriting {len(_existing)} existing clips in {DATA_DIR}')
        extract_clips_from_selections(RAW_AUDIO_DIR, SELECTIONS_DIR, DATA_DIR)

In [ ]:
# Cell 9: Training Infrastructure

def compute_class_weights(file_names, class_to_idx, num_labels=None):
    """Inverse-frequency class weights, one per MODEL OUTPUT class.

    num_labels must be the classifier head size (config['num_classes']), not
    len(class_to_idx): the detector folds 8 bat species + 'noise' into 2
    labels, so keying the vector off the dict length yields a 9-element
    tensor for a 2-output head and CrossEntropyLoss rejects it.
    """
    labels = [class_to_idx[get_class_from_filename(f)] for f in file_names]
    counts = Counter(labels)
    total = len(labels)
    n = num_labels if num_labels else len(class_to_idx)
    weights = torch.zeros(n)
    n_present = len(counts)
    for idx, count in counts.items():
        weights[idx] = total / (n_present * count) if count else 0.0
    return weights

def make_weighted_sampler(file_names, class_to_idx, num_labels=None):
    """Create WeightedRandomSampler for class balancing.

    Consequence to keep in mind: sampling with weight 1/count makes every class appear with
    probability exactly 1/n_classes. That is what we want for gradient balance, but it means
    the model's output distribution reflects a UNIFORM class prior, not the dataset's. The
    detector gate in Cell 14 corrects for this; see the CLASS PRIORS block in Cell 7.
    """
    labels = [class_to_idx[get_class_from_filename(f)] for f in file_names]
    weights = compute_class_weights(file_names, class_to_idx, num_labels)
    sample_weights = [weights[l] for l in labels]
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

def train_model(model, train_dataset, val_dataset, config, device, save_path=None):
    """
    Fine-tune `model` on windowed clips. AMP, optional gradient accumulation, DataParallel.

    train_dataset : WindowedBatDataset(train=True)  -> a new random 20 ms crop per clip per epoch
    val_dataset   : WindowedBatDataset(train=False) -> clip-level score = softmax averaged over
                    that clip's top-k windows (same protocol used for the test set)
    Model selection / LR schedule / early stopping all use SELECT_METRIC on the validation
    set (balanced accuracy by default: plain accuracy cannot tell a real detector from an
    "always predict call" one on an 80/20 split).

    save_path : if given, the training curves are written there as a PNG. It has NO other
                role -- see the note below.

    Returns (model, history, best_score).
    """
    # Weighted sampler for balanced batches (50/50 per class).
    # NOTE: We do NOT also pass class_weights to CrossEntropyLoss.
    # WeightedRandomSampler already balances; adding loss weights on top
    # creates a 4x noise penalty that collapses m03 (weak pretrain) to all-noise.
    all_file_names = train_dataset.file_names
    class_to_idx = train_dataset.class_to_idx
    sampler = make_weighted_sampler(all_file_names, class_to_idx,
                                    config.get('num_classes'))

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'],
                              sampler=sampler, num_workers=4, pin_memory=True,
                              persistent_workers=True)

    # Loss, optimizer, scheduler
    loss_fn = nn.CrossEntropyLoss()  # unweighted; sampler already balances batches
    effective_lr = config['base_lr']  # paper LR is absolute; do NOT scale by batch size
    optimizer = optim.Adam(model.parameters(), lr=effective_lr, betas=(config.get('beta1', 0.5), 0.999))
    patience_lr = int(max(1, ceil(config.get('lr_patience_epochs', 8) / config['epochs_per_eval'])))
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=patience_lr, factor=config.get('lr_decay_factor', 0.5), threshold=1e-3, threshold_mode='abs')
    patience_early = int(config.get('early_stopping_patience_epochs', 20))  # RAW epochs, read from config
    grad_accum_steps = config.get('grad_accum_steps', 1)
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    # Multi-GPU
    is_dp = config.get('use_multi_gpu', False) and torch.cuda.device_count() > 1
    if is_dp:
        model = nn.DataParallel(model)
        print(f'DataParallel across {torch.cuda.device_count()} GPUs')

    print(f'Grad accum: {grad_accum_steps}, AMP: {use_amp}, DP: {is_dp}, '
          f'batch: {config["batch_size"]} ({len(train_loader)} steps/epoch)')
    print(f'Effective LR: {effective_lr:.2e}   selection metric: {SELECT_METRIC}')
    print(f'Patience LR: {patience_lr} validations, Patience ES: {patience_early} raw epochs')

    history = {'train_loss': [], 'val_acc': [], 'val_bal_acc': [], 'val_score': [],
               'val_epoch': [], 'lr': []}
    # FIX: 0.0 meant a first validation scoring exactly 0.0 would never beat it, so best_state
    # would never be captured and the returned "best score" would be a floor, not a score.
    best_score = -np.inf
    best_epoch = None
    best_state = None
    no_improve = 0

    for epoch in range(config['n_epochs']):
        # Train
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        optimizer.zero_grad()

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config["n_epochs"]}', leave=False)
        for step, (inputs, labels) in enumerate(pbar):
            inputs, labels = inputs.to(device), labels.to(device)
            with torch.amp.autocast(device.type, enabled=use_amp):
                output = model(inputs)
                loss = loss_fn(output, labels) / grad_accum_steps
            scaler.scale(loss).backward()
            if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            epoch_loss += loss.item() * grad_accum_steps
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        history['train_loss'].append(avg_loss)

        # Validate every epochs_per_eval
        if (epoch + 1) % config['epochs_per_eval'] == 0 or epoch == config['n_epochs'] - 1:
            val_probs, val_labels = predict_proba(model, val_dataset, device)
            val_pred = val_probs.argmax(axis=1)
            val_acc = accuracy_score(val_labels, val_pred)
            val_bal = balanced_accuracy_score(val_labels, val_pred)
            # FIX (Space Bunny Free): this used to be
            #     val_score = val_bal if SELECT_METRIC == 'balanced_accuracy' else val_acc
            # which falls back to plain accuracy for ANY other string, with no diagnostic. A
            # one-character typo ('balanced_acc') silently reinstated the exact bug the metric
            # switch exists to prevent, while the log still printed "selection metric:
            # balanced_acc". Cell 2 validates the key at config time; this is a dict lookup,
            # so an unhandled value is now a KeyError rather than a silent behaviour change.
            val_score = {'balanced_accuracy': val_bal, 'accuracy': val_acc}[SELECT_METRIC]
            history['val_acc'].append(val_acc)
            history['val_bal_acc'].append(val_bal)
            history['val_score'].append(val_score)
            history['val_epoch'].append(epoch + 1)
            history['lr'].append(optimizer.param_groups[0]['lr'])
            scheduler.step(val_score)

            if val_score > best_score:
                best_score = val_score
                best_epoch = epoch + 1
                no_improve = 0
                # FIX (Space Bunny Free): the capture was wrapped in `if save_path:`.
                # A PLOT FILENAME was deciding which weights the caller received: with
                # save_path=None -- the documented default of this signature -- best_state
                # stayed None, nothing was restored, and train_model returned the LAST-EPOCH
                # model while still reporting the BEST validation score. Executed proof on a
                # 30-epoch run, save_path the only difference:
                #     save_path set  -> returned weight magnitude 3.9687 (best checkpoint)
                #     save_path None  -> returned weight magnitude 7.6348 (last epoch)
                #     ...both reporting best score 1.0000.
                # Always bare keys (strip DataParallel's "module." prefix).
                _sd = model.module.state_dict() if is_dp else model.state_dict()
                best_state = {k: v.detach().clone() for k, v in _sd.items()}
            else:
                no_improve += config['epochs_per_eval']  # RAW epochs elapsed, not validation steps

            print(f'  Epoch {epoch+1}: loss={avg_loss:.4f} val_acc={val_acc:.4f} val_bal={val_bal:.4f} '
                  f'best_bal={best_score:.4f} lr={optimizer.param_groups[0]["lr"]:.2e}')

            if no_improve >= patience_early:
                print(f'Early stopping at epoch {epoch+1}.')
                break

    # Restore best weights -- UNCONDITIONAL.
    # FIX: this was `if save_path and best_state is not None`, i.e. the same filename gate.
    if best_state is not None:
        if is_dp:
            model.module.load_state_dict(best_state)
        else:
            model.load_state_dict(best_state)
    else:
        print('  WARNING: no validation ever improved on -inf; returning last-epoch weights.')

    # -------------------------------------------------------------------------
    # Selection optimism, made visible.
    # The returned checkpoint is the single argmax over up to n_epochs validations of a
    # DISCRETE metric on 145 clips. One flipped clip moves balanced accuracy by
    # 0.5*(1/117 + 1/28) = 0.0221, so the argmax sits a couple of clips above the run's
    # typical validation level. That is the whole content of the "should n_epochs be larger?"
    # argument: printing both numbers turns it into a measurement.
    # -------------------------------------------------------------------------
    topk_mean = None
    if REPORT_TOPK_MEAN and history['val_score']:
        _k = min(int(REPORT_TOPK_MEAN), len(history['val_score']))
        topk_mean = float(np.mean(sorted(history['val_score'])[-_k:]))
        _mean_all = float(np.mean(history['val_score']))
        # How many clips is the argmax-topk gap worth? Needs the val class counts, which only
        # a dataset that carries labels can supply -- this is a diagnostic, so never require it.
        _quantum = None
        _vlab = getattr(val_dataset, 'labels', None)
        if _vlab is not None and len(_vlab):
            _u, _c = np.unique(np.asarray(_vlab), return_counts=True)
            if len(_u) > 1:
                _quantum = 0.5 * float(np.sum(1.0 / _c)) / len(_u)
        _gap = best_score - topk_mean
        print(f'  selection: argmax {best_score:.4f} @epoch {best_epoch} | '
              f'mean of top-{_k} evals {topk_mean:.4f} | mean of all {_mean_all:.4f} | '
              f'argmax-topk gap {_gap:+.4f}'
              + (f' = {_gap / _quantum:.1f} val clips' if _quantum else ''))
    history['best_score'] = best_score
    history['best_epoch'] = best_epoch
    history['topk_mean'] = topk_mean

    # FIX: save_path used to be the best_state gate AND nothing else -- there was no savefig
    # anywhere in the notebook, so the *_curves.png names were fiction. It writes a plot now.
    if save_path:
        try:
            _ve = history['val_epoch']
            fig, ax = plt.subplots(1, 3, figsize=(15, 4))
            ax[0].plot(history['train_loss'], color='C0')
            ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].set_title('train loss')
            ax[0].axvline(best_epoch, ls=':', c='k')
            ax[1].plot(_ve, history['val_acc'], label='accuracy', color='C1')
            ax[1].plot(_ve, history['val_bal_acc'], label='balanced accuracy', color='C2')
            ax[1].set_xlabel('epoch'); ax[1].set_title('validation'); ax[1].legend(fontsize=8)
            if topk_mean is not None:
                ax[1].axhline(topk_mean, ls='--', c='k', label=f'mean of top-{REPORT_TOPK_MEAN}')
            ax[2].plot(_ve, history['lr'], color='C3')
            ax[2].set_xlabel('epoch'); ax[2].set_title('learning rate'); ax[2].set_yscale('log')
            fig.suptitle(f'{SELECT_METRIC}: argmax {best_score:.4f} @ {best_epoch}'
                         + (f' | top-{REPORT_TOPK_MEAN} mean {topk_mean:.4f}' if topk_mean else ''),
                         fontsize=10)
            fig.tight_layout()
            fig.savefig(save_path, dpi=110)
            plt.close(fig)
            print(f'  curves -> {save_path}')
        except Exception as e:                          # noqa: BLE001 - never fail a run over a plot
            print(f'  (could not write curves to {save_path}: {e!r})')

    return model, history, best_score

print('Training infrastructure defined')


In [ ]:
# Cell 10: Load Pre-trained Models from the uploaded Kaggle dataset
#
# Expects the official BatSpot models (extracted from BatSpot_article):
#   <dataset>/batspot/models_call_detector/m03|m09|m11/train/ANIMAL-SPOT.pk
#   <dataset>/batspot/models_call_classifier/m09/train/ANIMAL-SPOT.pk
#
# NOTE: all four files share the basename ANIMAL-SPOT.pk, so each is staged
# here under a UNIQUE name. Copying by basename would let the classifier
# overwrite the detector and silently fine-tune the detector from the
# 15-class classifier weights.

os.makedirs(PRETRAINED_DIR, exist_ok=True)

# --- Priority 1: explicit paths from Cell 2 ---
_official = {}
_manual_set = set()
_missing_manual = []
_manual_set_attempted = False
if USE_MANUAL_MODEL_PATHS:
    for _lbl, _role, _mic, _p in [
            ('detector m03', 'detector', 'm03', DETECTOR_M03_PATH),
            ('detector m09', 'detector', 'm09', DETECTOR_M09_PATH),
            ('detector m11', 'detector', 'm11', DETECTOR_M11_PATH),
            ('classifier m09', 'classifier', 'm09', CLASSIFIER_M09_PATH)]:
        if _p is None:
            continue
        _manual_set_attempted = True
        if os.path.isfile(_p):
            _ap = os.path.abspath(_p)
            _manual_set.add(_ap)
            _official.setdefault(_role, {})[_mic] = _ap
        else:
            _missing_manual.append(f'{_lbl}: {_p}')
    if _official:
        print(f'Using {sum(len(v) for v in _official.values())} manual path(s) from Cell 2.')
    for _m in _missing_manual:
        print(f'  manual path NOT FOUND -> {_m}')
    if _missing_manual:
        print('  falling back to auto-discovery for the missing ones.')
else:
    print('USE_MANUAL_MODEL_PATHS = False -> ignoring Cell 2 paths.')

# --- Priority 2: discover the dataset directory (local repo or /kaggle/input/*) ---
BATSPOT_ARTICLE_DIR = None
_MODEL_GLOB = os.path.join('models_*', '*', 'train', 'ANIMAL-SPOT.pk')

for _cand in ['BatSpot_article/batspot', 'BatSpot_article']:
    if glob.glob(os.path.join(_cand, _MODEL_GLOB)):
        BATSPOT_ARTICLE_DIR = _cand
        break

if BATSPOT_ARTICLE_DIR is None:
    for _root in sorted(glob.glob('/kaggle/input/*')):
        for _cand in [os.path.join(_root, 'batspot'), _root]:
            if glob.glob(os.path.join(_cand, _MODEL_GLOB)):
                BATSPOT_ARTICLE_DIR = _cand
                break
        if BATSPOT_ARTICLE_DIR:
            break

# --- Collect sources, keyed role -> mic -> path ---
#
# _official ALREADY holds whatever the manual block above resolved. Do NOT
# re-initialise it here: doing so discards the Cell 2 paths and makes this
# cell report "No pre-trained models found" even though the manual lookup
# just succeeded. Discovery only fills roles/mics the manual block missed.
_manual_n = sum(len(v) for v in _official.values())
_skipped = []
if BATSPOT_ARTICLE_DIR:
    for _p in sorted(glob.glob(os.path.join(BATSPOT_ARTICLE_DIR, _MODEL_GLOB))):
        _parts = os.path.relpath(_p, BATSPOT_ARTICLE_DIR).split(os.sep)
        # e.g. ['models_call_detector', 'm09', 'train', 'ANIMAL-SPOT.pk']
        if len(_parts) < 4:
            continue
        _family, _mic = _parts[0], _parts[1]
        # Resolve the role FIRST, and only ever to 'classifier' or 'detector'.
        # buzz_detector / social_detector use different band limits and are
        # not fine-tuned here; classifying them as detectors would feed their
        # weights into the call-detector slot.
        if _family == 'models_call_classifier':
            _role = 'classifier'
        elif _family == 'models_call_detector':
            _role = 'detector'
        else:
            _skipped.append(f'{_family}/{_mic}')
            continue
        if _mic in _official.get(_role, {}):
            continue  # already supplied manually via Cell 2 (manual wins)
        _official.setdefault(_role, {})[_mic] = _p
    if _manual_n:
        print(f'Auto-discovery added nothing (Cell 2 supplied {_manual_n} model(s)); '
              f'manual paths stand.')

# --- Stage under unique names and read REAL metadata from each .pk ---
# PRETRAINED_MODELS[role][mic] = {path, sr, classes, num_classes, data, source}
PRETRAINED_MODELS = {}
detector_path = None
classifier_path = None

if _official:
    if BATSPOT_ARTICLE_DIR:
        print(f'BatSpot_article dataset: {BATSPOT_ARTICLE_DIR}\n')
    for _role, _mics in _official.items():
        for _mic, _src in sorted(_mics.items()):
            _dest = os.path.join(PRETRAINED_DIR, f'official_{_role}_{_mic}.pk')
            shutil.copy2(_src, _dest)
            try:
                _o = torch.load(_dest, map_location='cpu', weights_only=False)
                _d = _o.get('dataOpts', {})
                PRETRAINED_MODELS.setdefault(_role, {})[_mic] = {
                    'path': _dest,
                    'data': _d,
                    'sr': _d.get('sr'),
                    'classes': _o.get('classes', {}),
                    'num_classes': _o.get('classifierOpts', {}).get('num_classes'),
                    'source': _src,
                    'origin': 'manual' if os.path.abspath(_src) in _manual_set else 'discovered',
                }
            except Exception as e:
                print(f'  WARNING: could not read {_dest}: {e}')
    print('Official models staged:')
    for _role in sorted(PRETRAINED_MODELS):
        for _mic, _info in sorted(PRETRAINED_MODELS[_role].items()):
            _sr = _info['sr'] or 0
            print(f'  {_role:9s} {_mic}: {_info["num_classes"]:2d} classes, {_sr//1000:3d} kHz'
                  f'  [{_info.get("origin", "?")}] <- {_info["source"]}')
    # m09 is the paper's primary microphone (call detector test F1 0.985).
    # Prefer it; fall back to the lowest-numbered variant.
    for _role in ('detector', 'classifier'):
        _mics = PRETRAINED_MODELS.get(_role, {})
        if not _mics:
            continue
        _pick = 'm09' if 'm09' in _mics else sorted(_mics)[0]
        if _role == 'detector':
            detector_path = _mics[_pick]['path']
        else:
            classifier_path = _mics[_pick]['path']
        print(f'  -> fine-tuning {_role} from {_pick}')
    if _skipped:
        print(f'  (skipped, not fine-tuned here: {sorted(set(_skipped))})')
elif not _official:
    print('No pre-trained models found.')
    if USE_MANUAL_MODEL_PATHS and _manual_set_attempted:
        print('All Cell 2 paths were set but none resolved to a readable file.')
        print('Re-run Cell 2 and check the per-path OK / NOT FOUND lines above it.')
    else:
        print('Either set DETECTOR_M03_PATH / DETECTOR_M09_PATH / DETECTOR_M11_PATH /')
        print('CLASSIFIER_M09_PATH in Cell 2, or attach the dataset via Add Data.')

# --- Verify our spectrogram config matches each official model's dataOpts ---
print('\nConfig vs model dataOpts check:')
for _role, _cfg in (('detector', DET_CONFIG), ('classifier', CLS_CONFIG)):
    _mics = PRETRAINED_MODELS.get(_role)
    if not _mics:
        continue
    for _mic, _info in sorted(_mics.items()):
        _d = _info['data']
        _bad = []
        for _k in ('sr', 'n_fft', 'hop_length', 'fmin', 'fmax', 'n_freq_bins', 'freq_compression'):
            if _k in _d and _k in _cfg and _d[_k] != _cfg[_k]:
                _bad.append(f'{_k}: model={_d[_k]} config={_cfg[_k]}')
        _tag = f'official {_role} ({_mic})'
        if _bad:
            print(f'  MISMATCH {_tag}: ' + '; '.join(_bad))
        else:
            print(f'  OK        {_tag}')

# --- Class-overlap report (official models use European species codes) ---
if 'classes' in dir() or 'classes' in globals():
    _mine = list(globals().get('classes', []))
    for _role, _cfg in (('detector', DET_CONFIG), ('classifier', CLS_CONFIG)):
        for _mic, _info in sorted(PRETRAINED_MODELS.get(_role, {}).items()):
            _theirs = set(_info['classes'].keys())
            _ov = sorted(set(_mine) & _theirs)
            print(f'\n{_role} ({_mic}) knows {len(_theirs)} classes; '
                  f'overlap with your data {sorted(_mine)}: {_ov or "NONE"}')
            if not _ov and _role == 'classifier':
                print('  -> classifier head cannot transfer; only the encoder will be used.')
            elif _ov and _role == 'detector':
                print('  -> evaluated on the overlapping classes only (noise).')

# --- Fallback: local Data/model_output (384 kHz, 3-class) ---
if detector_path is None and classifier_path is None:
    _local = None
    for _cand in ['Data/model_output/ANIMAL-SPOT.pk',
                  '/kaggle/input/batspot-data/model_output/ANIMAL-SPOT.pk']:
        if os.path.isfile(_cand):
            _local = _cand
            break
    if _local is None and MODEL_OUTPUT_DIR and os.path.isdir(MODEL_OUTPUT_DIR):
        for _root, _, _files in os.walk(MODEL_OUTPUT_DIR):
            for _f in _files:
                if _f.endswith('.pk'):
                    _local = os.path.join(_root, _f)
                    break
            if _local:
                break
    if _local:
        _dest = os.path.join(PRETRAINED_DIR, 'fallback_local.pk')
        shutil.copy2(_local, _dest)
        detector_path = _dest
        classifier_path = _dest
        print(f'\nFallback local model: {_local}')
        print('  NOTE: 384 kHz / 3-class model. Spectrogram params will NOT match')
        print('        DET_CONFIG or CLS_CONFIG, so encoder transfer is unreliable.')
    else:
        print('\nWARNING: no pre-trained model found. Both models train from scratch.')

print(f'\nFine-tuning starting points:')
print(f'  Detector:   {detector_path}')
print(f'  Classifier: {classifier_path}')

In [ ]:
# Cell 11: Build Detector Dataset (shared across all 3 detector variants)
#
# Uses the shared train_wavs / val_wavs / test_wavs split. Every non-noise
# species is folded to label 1 (call); noise is mapped to 0.

_det_all = list(train_wavs) + list(val_wavs) + list(test_wavs)
_det_species = sorted({get_class_from_filename(f) for f in _det_all})
if 'noise' not in _det_species:
    print("WARNING: no 'noise' files in the dataset; detector sees only calls.")

# All species except 'noise' -> label 1 (call)
det_class_to_idx = {c: (0 if c == 'noise' else 1) for c in _det_species}
det_class_to_idx.setdefault('noise', 0)
det_class_to_idx.setdefault('call', 1)
det_idx_to_class = {0: 'noise', 1: 'call'}

print(f'Detector: {len(_det_species)} dataset classes -> 2 binary labels')
print(f'  label 0 (noise): {[c for c, v in det_class_to_idx.items() if v == 0]}')
print(f'  label 1 (call):  {[c for c, v in det_class_to_idx.items() if v == 1]}')

# 20 ms windows: train = random loud crop per epoch, val/test = top-k windows averaged per clip
det_seq_len = int(float(DET_CONFIG['sequence_len']) / 1000 * DET_CONFIG['sr'] / DET_CONFIG['hop_length'])
print(f'Detector: sr={DET_CONFIG["sr"]}, window={det_seq_len} frames (20 ms), WINDOW_MODE={WINDOW_MODE!r}')

det_train = WindowedBatDataset(train_wavs, det_class_to_idx, DET_CONFIG, train=True,
                               augment=USE_AUGMENTATION)
det_val   = WindowedBatDataset(val_wavs,   det_class_to_idx, DET_CONFIG, train=False)
det_test  = WindowedBatDataset(test_wavs,  det_class_to_idx, DET_CONFIG, train=False)
print(f'Detector eval windows: val={len(det_val)} ({len(det_val.file_names)} clips), '
      f'test={len(det_test)} ({len(det_test.file_names)} clips)')

# Show binary class distribution
det_train_labels = [det_class_to_idx[get_class_from_filename(f)] for f in train_wavs]
det_val_labels   = [det_class_to_idx[get_class_from_filename(f)] for f in val_wavs]
det_test_labels  = [det_class_to_idx[get_class_from_filename(f)] for f in test_wavs]
print(f'Detector train: {Counter(det_train_labels)}  (0=noise, 1=call)')
print(f'Detector val:   {Counter(det_val_labels)}')
print(f'Detector test:  {Counter(det_test_labels)}')

In [ ]:
# Cell 12: Train ALL 3 detector variants (m03, m09, m11) + evaluate each individually.
#
# PRETRAINED_MODELS['detector'] = {'m03': {...}, 'm09': {...}, 'm11': {...}}
# For each mic: load its pretrained encoder, fine-tune, evaluate on det_test.
# Results stored in det_results[mic].

def evaluate_model(model, dataset, class_names, device, model_name='Model'):
    """Evaluate on a windowed eval dataset (clip-level, windows averaged). Returns metrics dict."""
    all_probs, all_labels = predict_proba(model, dataset, device)
    all_preds = all_probs.argmax(axis=1)
    eval_labels = list(range(len(class_names)))
    acc  = accuracy_score(all_labels, all_preds)
    bal  = balanced_accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0)
    report = classification_report(all_labels, all_preds, labels=eval_labels,
                                   target_names=class_names, zero_division=0,
                                   output_dict=True)
    cm = confusion_matrix(all_labels, all_preds, labels=eval_labels)
    print(f'\n=== {model_name} ===')
    print(f'Accuracy: {acc:.4f}  Balanced acc: {bal:.4f}  Precision: {prec:.4f}  '
          f'Recall: {rec:.4f}  F1: {f1:.4f}')
    print(classification_report(all_labels, all_preds, labels=eval_labels,
                                target_names=class_names, zero_division=0))
    return {'accuracy': acc, 'balanced_accuracy': bal, 'precision': prec, 'recall': rec, 'f1': f1,
            'predictions': all_preds, 'labels': all_labels, 'probs': all_probs,
            'confusion_matrix': cm, 'report': report, 'class_names': class_names}


encoderOpts_det = {'input_channels': 1, 'conv_kernel_size': 7, 'max_pool': 2, 'resnet_size': 18}
_det_mics = sorted(PRETRAINED_MODELS.get('detector', {}).keys())
if not _det_mics:
    print('No pretrained detector variants found -> training one from scratch.')
    _det_mics = ['scratch']

# det_results[mic] = {'model', 'metrics', 'best_val_acc', 'encoderOpts', 'classifierOpts'}
det_results = {}

for _mic in _det_mics:
    print(f'\n{"="*60}')
    print(f'DETECTOR VARIANT: {_mic}')
    print(f'{"="*60}')

    model, encoder, _cls_head, classifierOpts = build_model(
        encoderOpts_det, DET_CONFIG['num_classes'], DEVICE)

    _pk = PRETRAINED_MODELS.get('detector', {}).get(_mic, {}).get('path')
    if _pk:
        try:
            _pre, _, _pre_opts = load_model_from_pk(_pk, DEVICE)
            encoder.load_state_dict(dict(list(_pre.named_children())[0][1].state_dict()))
            print(f'Encoder loaded from {os.path.basename(_pk)} ({_pre_opts.get("sr", 0)//1000}kHz)')
        except Exception as _e:
            print(f'Could not load encoder for {_mic}: {_e}')
    else:
        print(f'No pretrained path for {_mic} -> random init.')

    model = model.to(DEVICE)
    print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

    model, _hist, best_acc = train_model(
        model, det_train, det_val, DET_CONFIG, DEVICE,
        save_path=os.path.join(WORKING_DIR, f'detector_{_mic}_curves.png'),
    )
    print(f'Best val {SELECT_METRIC} ({_mic}): {best_acc:.4f}')

    metrics = evaluate_model(model, det_test, DET_CLASSES, DEVICE,
                             f'Detector {_mic} (Test Set)')

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(metrics['confusion_matrix'], interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=range(2), yticks=range(2),
           xticklabels=DET_CLASSES, yticklabels=DET_CLASSES,
           xlabel='Predicted', ylabel='True',
           title=f'Detector {_mic} Confusion Matrix (Test)')
    for i in range(2):
        for j in range(2):
            v = metrics['confusion_matrix'][i, j]
            ax.text(j, i, str(v), ha='center', va='center',
                    color='white' if v > metrics['confusion_matrix'].max()/2 else 'black')
    plt.tight_layout(); plt.show()

    det_results[_mic] = {'model': model, 'metrics': metrics,
                         'best_val_acc': best_acc,
                         'encoderOpts': encoderOpts_det,
                         'classifierOpts': classifierOpts}

# Comparison table
print(f'\n{"="*60}')
print('DETECTOR COMPARISON SUMMARY')
print(f'{"="*60}')
print(f'{"Variant":<10} {"Val score":>9} {"Test Acc":>10} {"Bal Acc":>8} {"Prec":>8} {"Rec":>8} {"F1":>8}')
print('-'*70)
for _mic, _r in det_results.items():
    m = _r['metrics']
    print(f'{_mic:<10} {_r["best_val_acc"]:>9.4f} {m["accuracy"]:>10.4f} {m["balanced_accuracy"]:>8.4f} '
          f'{m["precision"]:>8.4f} {m["recall"]:>8.4f} {m["f1"]:>8.4f}')

In [ ]:
# Cell 13: Build Classifier Dataset + Train Classifier (single, shared across all pipelines)

cls_seq_len = int(float(CLS_CONFIG['sequence_len']) / 1000 * CLS_CONFIG['sr'] / CLS_CONFIG['hop_length'])
print(f'Classifier: sr={CLS_CONFIG["sr"]}, window={cls_seq_len} frames (20 ms), WINDOW_MODE={WINDOW_MODE!r}')

cls_train = WindowedBatDataset(train_wavs, CLS_CLASS_TO_IDX, CLS_CONFIG, train=True,
                               augment=USE_AUGMENTATION)
cls_val   = WindowedBatDataset(val_wavs,   CLS_CLASS_TO_IDX, CLS_CONFIG, train=False)
cls_test  = WindowedBatDataset(test_wavs,  CLS_CLASS_TO_IDX, CLS_CONFIG, train=False)

print(f'Classifier train: {Counter([CLS_CLASS_TO_IDX[get_class_from_filename(f)] for f in train_wavs])}')
print(f'Classifier val:   {Counter([CLS_CLASS_TO_IDX[get_class_from_filename(f)] for f in val_wavs])}')
print(f'Classifier test:  {Counter([CLS_CLASS_TO_IDX[get_class_from_filename(f)] for f in test_wavs])}')

encoderOpts_cls = {'input_channels': 1, 'conv_kernel_size': 7, 'max_pool': 2, 'resnet_size': 18}
cls_model, cls_encoder, cls_classifier_head, cls_classifierOpts = build_model(
    encoderOpts_cls, CLS_CONFIG['num_classes'], DEVICE)

if classifier_path:
    try:
        _pre, _, _pre_opts = load_model_from_pk(classifier_path, DEVICE)
        cls_encoder.load_state_dict(dict(list(_pre.named_children())[0][1].state_dict()))
        print(f'Encoder loaded from {os.path.basename(classifier_path)} ({_pre_opts.get("sr",0)//1000}kHz)')
    except Exception as _e:
        print(f'Could not load classifier pretrained: {_e}')
else:
    print('No pre-trained classifier. Training from scratch.')

cls_model = cls_model.to(DEVICE)
print(f'Classifier params: {sum(p.numel() for p in cls_model.parameters()):,}')

cls_model, cls_history, cls_best_acc = train_model(
    cls_model, cls_train, cls_val, CLS_CONFIG, DEVICE,
    save_path=os.path.join(WORKING_DIR, 'classifier_curves.png'),
)
print(f'\nClassifier training complete. Best val {SELECT_METRIC}: {cls_best_acc:.4f}')

cls_metrics = evaluate_model(cls_model, cls_test, CLS_CLASSES, DEVICE, 'Classifier (Test Set)')

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cls_metrics['confusion_matrix'], interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
n_cls = len(CLS_CLASSES)
ax.set(xticks=range(n_cls), yticks=range(n_cls),
       xticklabels=CLS_CLASSES, yticklabels=CLS_CLASSES,
       xlabel='Predicted', ylabel='True', title='Classifier Confusion Matrix (Test)')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
for i in range(n_cls):
    for j in range(n_cls):
        v = cls_metrics['confusion_matrix'][i, j]
        ax.text(j, i, str(v), ha='center', va='center',
                color='white' if v > cls_metrics['confusion_matrix'].max()/2 else 'black')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 14: Combined Pipeline — each detector variant (m03/m09/m11) x the single classifier.
#
# All probabilities are clip-level (softmax averaged over the clip's top-k windows).
# Four ways of combining detector + classifier are reported side by side, because on this
# dataset the classifier already has a 'noise' class and a hard detector gate can only ADD
# errors:
#   classifier alone            : argmax of the 8-class classifier (no detector)
#   hard gate (raw)             : P(call) < val-tuned threshold -> 'noise', else classifier
#   hard gate (prior-corrected) : same, on probabilities rescaled to the evaluation prior
#   soft combine                : average the detector's and classifier's noise evidence
#
# WHY THE PRIOR-CORRECTED VARIANT EXISTS (Space Bunny Free)
# The sampler draws each class with probability exactly 1/2, so the detector is trained under
# a 50/50 noise/call prior while val/test is ~19/81. Its outputs are therefore shifted toward
# 'call' by (0.807/0.5)/(0.193/0.5) ~ 4.2 in odds. That is exactly what the Kaggle run showed:
# m09 with noise PRECISION 0.70 against recall 0.93, and val-tuned thresholds that had to sink
# to 0.30 / 0.40 / 0.10 to compensate. Such a threshold bakes in the 19 % noise prior and will
# NOT transfer to real recordings, where noise is >95 % of windows.
# Rescaling by prior_eval/prior_train makes P(call) > 0.5 mean what it says. Both variants are
# printed so the numbers stay comparable with the earlier Kaggle run.

def prior_correct_probs(probs, prior_eval, prior_train):
    """Rescale class probabilities from the training prior to the evaluation prior.

    Bayes:  p_eval(c|x) ∝ p_train(c|x) · prior_eval(c) / prior_train(c)
    """
    adj = np.asarray(probs, dtype=np.float64) * (np.asarray(prior_eval, float) /
                                                 np.asarray(prior_train, float))
    adj = np.clip(adj, 0.0, None)
    return adj / np.maximum(adj.sum(axis=1, keepdims=True), 1e-12)


def _tune_threshold(val_call, val_true, grid=np.arange(0.05, 0.96, 0.05)):
    """Threshold maximising call-class F1 on the VALIDATION set. Never uses test."""
    best_t, best_f1 = float(grid[0]), -1.0
    for t in grid:
        _, _, f1, _ = precision_recall_fscore_support(
            val_true, (val_call >= t).astype(int), average='binary', pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, best_f1


def run_combined_pipeline(det_model, mic_label, cls_model, det_val, det_test, cls_test, device):
    """Tune thresholds on val, then evaluate the four variants on test."""
    if [os.path.basename(f) for f in det_test.file_names] != \
       [os.path.basename(f) for f in cls_test.file_names]:
        raise ValueError('detector/classifier test file order differs')
    noise_idx = CLS_CLASS_TO_IDX['noise']

    # --- validation: tune both thresholds on val only -------------------------------------
    val_probs_raw, val_true = predict_proba(det_model, det_val, device)
    val_probs_corr = prior_correct_probs(val_probs_raw, DET_VAL_PRIOR, DET_EFFECTIVE_TRAIN_PRIOR)
    t_raw, f1_raw = _tune_threshold(val_probs_raw[:, 1], val_true)
    t_corr, f1_corr = _tune_threshold(val_probs_corr[:, 1], val_true)
    print(f'  [{mic_label}] raw threshold={t_raw:.2f} val-F1={f1_raw:.4f}   |   '
          f'prior-corrected threshold={t_corr:.2f} val-F1={f1_corr:.4f}')

    # --- test ------------------------------------------------------------------------------
    det_p_raw, _ = predict_proba(det_model, det_test, device)
    det_p_corr = prior_correct_probs(det_p_raw, DET_EVAL_PRIOR, DET_EFFECTIVE_TRAIN_PRIOR)
    cls_p, clabels = predict_proba(cls_model, cls_test, device)

    alone = cls_p.argmax(axis=1)
    gate_raw = np.where(det_p_raw[:, 1] < t_raw, noise_idx, alone)
    gate_corr = np.where(det_p_corr[:, 1] < t_corr, noise_idx, alone)
    p_noise = 0.5 * cls_p[:, noise_idx] + 0.5 * (1 - det_p_corr[:, 1])
    rest = cls_p.copy(); rest[:, noise_idx] = 0
    rest = rest / np.maximum(rest.sum(axis=1, keepdims=True), 1e-9) * (1 - p_noise)[:, None]
    rest[:, noise_idx] = p_noise
    soft = rest.argmax(axis=1)

    variants = (('classifier alone', alone),
                ('hard gate (raw)', gate_raw),
                ('hard gate (prior-corrected)', gate_corr),
                ('soft combine', soft))
    out = {'threshold': t_raw, 'threshold_prior_corrected': t_corr,
           'val_f1': f1_raw, 'val_f1_prior_corrected': f1_corr,
           'detector_noise_precision_at_0.5': float(
               precision_recall_fscore_support(
                   _det_test_labels(det_test, device), (det_p_raw[:, 1] >= 0.5).astype(int),
                   average='binary', pos_label=0, zero_division=0)[0]),
           'variants': {}}
    for name, preds in variants:
        prec, rec, f1, _ = precision_recall_fscore_support(
            clabels, preds, average='macro', zero_division=0)
        out['variants'][name] = {
            'acc': accuracy_score(clabels, preds), 'prec': prec, 'rec': rec, 'f1': f1,
            'cm': confusion_matrix(clabels, preds, labels=list(range(len(CLS_CLASSES)))),
            'report': classification_report(clabels, preds,
                                           labels=list(range(len(CLS_CLASSES))),
                                           target_names=CLS_CLASSES, zero_division=0)}
    # keep the original keys (raw hard gate) so older downstream code keeps working
    g = out['variants']['hard gate (raw)']
    out.update({'acc': g['acc'], 'prec': g['prec'], 'rec': g['rec'], 'f1': g['f1'],
                'cm': g['cm'], 'report': g['report']})
    return out


def _det_test_labels(det_test, device):
    """Binary noise/call labels for the detector test split (index 0 = noise)."""
    return np.asarray(det_test.labels)


pipeline_results = {}
for _mic, _dr in det_results.items():
    print(f'\n{"="*60}')
    print(f'COMBINED PIPELINE: Detector {_mic} -> Classifier')
    print(f'{"="*60}')
    _res = run_combined_pipeline(_dr['model'], _mic, cls_model, det_val, det_test, cls_test, DEVICE)
    pipeline_results[_mic] = _res

    _best = max(_res['variants'].items(), key=lambda kv: kv[1]['acc'])
    print(f'\n=== Combined {_mic} — best variant: {_best[0]} '
          f'(acc={_best[1]["acc"]:.4f} F1={_best[1]["f1"]:.4f}) ===')
    print(_best[1]['report'])

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(_best[1]['cm'], interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    n_cls = len(CLS_CLASSES)
    ax.set(xticks=range(n_cls), yticks=range(n_cls),
           xticklabels=CLS_CLASSES, yticklabels=CLS_CLASSES,
           xlabel='Predicted', ylabel='True',
           title=f'Combined {_mic} ({_best[0]}) Confusion Matrix (Test)')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    for i in range(n_cls):
        for j in range(n_cls):
            v = _best[1]['cm'][i, j]
            ax.text(j, i, str(v), ha='center', va='center',
                    color='white' if v > _best[1]['cm'].max() / 2 else 'black')
    plt.tight_layout(); plt.show()

# Final comparison table
print(f'\n{"="*92}')
print('COMBINED PIPELINE COMPARISON SUMMARY (test accuracy, 8 classes)')
print(f'{"="*92}')
print(f'{"Detector":<9} {"t_raw":>6} {"t_corr":>7} {"Clf alone":>10} {"Gate raw":>9} '
      f'{"Gate corr":>11} {"Soft":>7}')
print('-'*92)
for _mic, _r in pipeline_results.items():
    v = _r['variants']
    print(f'{_mic:<9} {_r["threshold"]:>6.2f} {_r["threshold_prior_corrected"]:>7.2f} '
          f'{v["classifier alone"]["acc"]:>10.4f} {v["hard gate (raw)"]["acc"]:>9.4f} '
          f'{v["hard gate (prior-corrected)"]["acc"]:>11.4f} {v["soft combine"]["acc"]:>7.4f}')
print('-'*92)
_best_mic = max(pipeline_results.items(),
                key=lambda kv: kv[1]['variants']['classifier alone']['acc'])[0]
print(f'Read this as: the gate columns differ from "Clf alone" only by the detector\'s own '
      f'errors. The classifier already models "noise", so the detector cannot add information '
      f'here -- it is still the right tool for SCANNING long recordings, not for labelling '
      f'pre-cut clips.')
print(f'Also note t_raw ({pipeline_results[_best_mic]["threshold"]:.2f}) vs t_corr '
      f'({pipeline_results[_best_mic]["threshold_prior_corrected"]:.2f}): the raw threshold was '
      f'fitted to compensate for the sampler\'s 50/50 prior. Use the prior-corrected one when '
      f'the noise fraction differs from the test split (i.e. on real recordings).')


In [ ]:
# Cell 15: Export Models as .pk
#
# Exports all fine-tuned detector variants + the single classifier.

def export_pk(model, encoderOpts, classifierOpts, dataOpts, classes, path):
    """Export model as .pk file compatible with BatSpot GUI/CLI."""
    if isinstance(model, nn.DataParallel):
        model = model.module
    model = model.cpu()
    encoder = model[0]
    classifier = model[1]
    torch.save({
        'encoderOpts':     encoderOpts,
        'classifierOpts':  classifierOpts,
        'dataOpts':        dataOpts,
        'encoderState':    encoder.state_dict(),
        'classifierState': classifier.state_dict(),
        'classes':         classes,
    }, path)
    print(f'Saved: {path} ({os.path.getsize(path)/1e6:.1f} MB)')

det_dataOpts = {
    'sr': DET_CONFIG['sr'], 'n_fft': DET_CONFIG['n_fft'],
    'hop_length': DET_CONFIG['hop_length'], 'n_freq_bins': DET_CONFIG['n_freq_bins'],
    'fmin': DET_CONFIG['fmin'], 'fmax': DET_CONFIG['fmax'],
    'freq_compression': DET_CONFIG['freq_compression'],
    'min_level_db': -100, 'ref_level_db': 20, 'preemphases': 0.98,
}
det_classes = {'noise': 0, 'call': 1}

# Export each detector variant
det_out_paths = {}
for _mic, _dr in det_results.items():
    _out = os.path.join(WORKING_DIR, f'detector_192khz_{_mic}.pk')
    export_pk(_dr['model'], _dr['encoderOpts'], _dr['classifierOpts'],
              det_dataOpts, det_classes, _out)
    det_out_paths[_mic] = _out

# Export classifier (250kHz, multi-species)
cls_dataOpts = {
    'sr': CLS_CONFIG['sr'], 'n_fft': CLS_CONFIG['n_fft'],
    'hop_length': CLS_CONFIG['hop_length'], 'n_freq_bins': CLS_CONFIG['n_freq_bins'],
    'fmin': CLS_CONFIG['fmin'], 'fmax': CLS_CONFIG['fmax'],
    'freq_compression': CLS_CONFIG['freq_compression'],
    'min_level_db': -100, 'ref_level_db': 20, 'preemphases': 0.98,
}
cls_out_path = os.path.join(WORKING_DIR, 'classifier_250khz.pk')
export_pk(cls_model, encoderOpts_cls, cls_classifierOpts, cls_dataOpts, CLS_CLASS_TO_IDX, cls_out_path)

# Verify exports
print('\nVerifying exports...')
for _mic, p in det_out_paths.items():
    obj = torch.load(p, map_location='cpu', weights_only=False)
    print(f'  detector_{_mic}: enc={len(obj["encoderState"])} params, '
          f'cls={len(obj["classifierState"])} params, classes={obj["classes"]}')
obj = torch.load(cls_out_path, map_location='cpu', weights_only=False)
print(f'  classifier:    enc={len(obj["encoderState"])} params, '
      f'cls={len(obj["classifierState"])} params, classes={list(obj["classes"].keys())}')

print('\nNOTE: these models were trained with per-window MIN-MAX normalisation on 20 ms windows,')
print('exactly like the official BatSpot models. When predicting with PREDICTION/start_prediction.py')
print('(or the GUI) enable min-max normalisation (min_max_norm=true), otherwise inputs are scaled differently.')

In [ ]:
# Cell 16: Model Inventory — official pre-trained models + your own models

if MODEL_OUTPUT_DIR is None:
    for candidate in ['Data/model_output', '/kaggle/input/model_output',
                      '/kaggle/input/batspot-data/model_output']:
        if os.path.isdir(candidate):
            MODEL_OUTPUT_DIR = candidate
            break

model_inventory = []
model_tags = {}

if MODEL_OUTPUT_DIR and os.path.isdir(MODEL_OUTPUT_DIR):
    for root, dirs, files in os.walk(MODEL_OUTPUT_DIR):
        for f in files:
            if f.endswith('.pk') or f.endswith('.checkpoint'):
                p = os.path.join(root, f)
                model_inventory.append(p)
                model_tags[p] = 'yours'

# Official models staged by Cell 10
for role, mics in PRETRAINED_MODELS.items():
    for mic, info in sorted(mics.items()):
        p = info['path']
        if p not in model_inventory:
            model_inventory.append(p)
        model_tags[p] = f'official:{role}/{mic}'

print(f'MODEL_OUTPUT_DIR: {MODEL_OUTPUT_DIR}')
print(f'Found {len(model_inventory)} models:\n')
for m in model_inventory:
    tag = model_tags.get(m, '?')
    size_mb = os.path.getsize(m) / 1e6
    fmt = '.pk' if m.endswith('.pk') else '.checkpoint'
    info = None
    for role, mics in PRETRAINED_MODELS.items():
        for mic, inf in mics.items():
            if inf['path'] == m:
                info = inf
    if info:
        print(f'  [{tag:22s}] {size_mb:6.1f} MB {fmt}  '
              f'{info["num_classes"]} classes, {(info["sr"] or 0)//1000} kHz')
    else:
        print(f'  [{tag:22s}] {size_mb:6.1f} MB {fmt}  {os.path.basename(m)}')


In [ ]:
# Cell 17: Load & Evaluate Each Existing Model on the TEST split
#
# WHY THIS CELL CHANGED: the previous version only scored files whose species name appeared
# in the model's own class list. The official detectors know {'noise','target'} and the official
# classifier 15 European species, so only the 186 'noise' files matched -- the reported
# "accuracy" (e.g. m03 = 0.984) was NOISE RECALL on all clips, i.e. "it says noise to almost
# everything". That number is not comparable to a fine-tuned model's accuracy on calls + noise.
#
# Now every model is scored on the SAME held-out test clips with the SAME windowed protocol:
#   * detector {noise,target}         : every bat species counts as 'target' -> call-vs-noise
#   * classifier w/o species overlap  : only "does it say noise?" is meaningful -> scored as a
#                                       noise-vs-call detector (species accuracy is N/A)
#   * model whose class names overlap : exact-name multi-class on the overlapping classes
#
# REVIEW NOTE (Space Bunny Free): the third branch still evaluates a reduced denominator by
# construction -- it can only use clips whose species the model knows, which is 79 of the 145
# test clips for the local 3-class models. That is inherent, not a bug, but it means those
# rows must never be placed in the same column as 145-clip rows without saying so. The `kind`
# and `n_files` fields below exist so the summary table can group them.
from sklearn.metrics import roc_auc_score


def run_model_evaluation(model_path, wav_files, device):
    """Evaluate one existing model on `wav_files` (pass test_wavs)."""
    model_name = os.path.basename(model_path)
    try:
        model, model_classes, dataOpts = load_any_model(model_path, device)
    except Exception as e:
        return {'name': model_name, 'path': model_path, 'error': str(e)}
    model.to(device).eval()

    data_species = {get_class_from_filename(f) for f in wav_files} - {'noise'}
    # FIX: read n_freq_bins the way predict.py does. The old `dataOpts.get('n_freq_bins', 256)`
    # silently used 256 for any model that stored `num_mels` instead, feeding it a spectrogram
    # of the wrong width -- a wrong number with no diagnostic.
    n_freq_bins = dataOpts.get('n_freq_bins') or dataOpts.get('num_mels') or 256
    fc = dataOpts.get('freq_compression', 'linear')
    if fc != 'linear':
        print(f'  WARNING: {model_name} was trained with freq_compression={fc!r}, but '
              f'clip_to_db_spectrogram only implements linear (nearest) band cropping. '
              f'The score below is NOT comparable to its training distribution.')
    cfg = {'sr': dataOpts.get('sr', 384000), 'n_fft': dataOpts.get('n_fft', 256),
           'hop_length': dataOpts.get('hop_length', 128),
           'n_freq_bins': n_freq_bins,
           'fmin': dataOpts.get('fmin', 18000), 'fmax': dataOpts.get('fmax', 90000),
           'sequence_len': 20}

    if set(model_classes) == {'noise', 'target'}:
        kind = 'detector: call vs noise'
        files = list(wav_files)
        label_map = {'noise': 0}
        label_map.update({c: 1 for c in data_species})
        names = ['noise', 'call']
    elif 'noise' in model_classes and not (set(model_classes) & data_species):
        kind = 'classifier as noise-vs-call ONLY (no species overlap)'
        files = list(wav_files)
        label_map = {'noise': 0}
        label_map.update({c: 1 for c in data_species})
        names = ['noise', 'call']
    else:
        kind = 'multi-class, overlapping species ONLY'
        files = [f for f in wav_files if get_class_from_filename(f) in model_classes]
        label_map = {get_class_from_filename(f): model_classes[get_class_from_filename(f)]
                     for f in files}
        names = None      # built from the probability width below

    if not files:
        return {'name': model_name, 'path': model_path, 'unsupported': True,
                'error': f'no overlapping files: model knows {sorted(model_classes)}, '
                         f'data has {sorted(data_species | {"noise"})}'}

    ds = WindowedBatDataset(files, label_map, cfg, train=False)
    probs, labels = predict_proba(model, ds, device)
    C = probs.shape[1]
    if names is None:
        # FIX: names used to be built by sorting model_classes by value, so a class dict with
        # non-contiguous indices produced fewer names than the model has outputs and
        # confusion_matrix(..., labels=range(len(names))) then indexed y_true*n_labels+y_pred
        # out of range and failed on reshape. Index-by-output-width instead: len(names) == C
        # always.
        inv = {v: k for k, v in model_classes.items()}
        names = [inv.get(i, f'class{i}') for i in range(C)]
        if len(model_classes) != C:
            print(f'  WARNING: {model_name} has {C} outputs but {len(model_classes)} class names.')

    out = {'name': model_name, 'path': model_path, 'kind': kind, 'n_files': len(files),
           'target_names': names, 'labels': labels, 'files': files, 'auc': None}
    # Class indices are resolved BY NAME, never assumed to be 0/1. The official classifier has
    # 'noise' at index 8 of 15, so a hard-coded [:, 1] would have been silently wrong.
    if kind.startswith('detector'):
        p_call = probs[:, model_classes['target']]
    elif kind.startswith('classifier'):
        p_call = 1.0 - probs[:, model_classes['noise']]
    else:
        p_call = None
    if p_call is not None:
        preds = (p_call >= 0.5).astype(int)
        out['auc'] = float(roc_auc_score(labels, p_call)) if len(set(labels)) > 1 else None
    else:
        preds = probs.argmax(axis=1)
    out.update({'preds': preds, 'accuracy': accuracy_score(labels, preds),
                'balanced_accuracy': balanced_accuracy_score(labels, preds),
                'confusion_matrix': confusion_matrix(labels, preds, labels=list(range(C))),
                'model_classes': model_classes})
    # AUC indistinguishable from chance -> no usable signal, whatever the accuracy says.
    if out['auc'] is not None and abs(out['auc'] - 0.5) < AUC_NO_SIGNAL:
        out['no_signal'] = True
    return out


# Evaluate all models on the held-out TEST split
val_results = []
for m_path in model_inventory:
    result = run_model_evaluation(m_path, test_wavs, DEVICE)
    val_results.append(result)
    tag = model_tags.get(m_path, '?')
    if 'error' in result:
        status = 'UNSUPPORTED' if result.get('unsupported') else 'ERROR'
        print(f'[{status}] {tag} / {result["name"]}: {result["error"]}')
    else:
        _auc = f', auc={result["auc"]:.4f}' if result['auc'] is not None else ''
        _ns = '  [NO SIGNAL: AUC ~ 0.5]' if result.get('no_signal') else ''
        print(f'[OK] {tag} / {result["name"]}: {result["kind"]}: acc={result["accuracy"]:.4f}, '
              f'bal_acc={result["balanced_accuracy"]:.4f}{_auc}, files={result["n_files"]}{_ns}')


In [ ]:
# Cell 18: Results Summary & Confusion Matrices

print('\n' + '='*80)
print('EXISTING MODEL EVALUATION SUMMARY (held-out test split)')
print('='*80)

for r in val_results:
    if 'error' in r:
        print(f'\n{r["name"]}: ERROR - {r["error"]}')
        continue
    print(f'\n--- {r["name"]} ---')
    print(f'  Mode: {r["kind"]}')
    print(f'  Files evaluated: {r["n_files"]} of {len(test_wavs)} test clips'
          + ('   <-- reduced denominator, NOT comparable to full-split rows'
             if r['n_files'] != len(test_wavs) else ''))
    print(f'  Accuracy: {r["accuracy"]:.4f}   Balanced accuracy: {r["balanced_accuracy"]:.4f}'
          + (f'   AUC: {r["auc"]:.4f}' if r['auc'] is not None else ''))
    if r.get('no_signal'):
        print('  *** AUC is within {:.2f} of 0.5: this model carries NO usable signal on this '
              'task.'.format(AUC_NO_SIGNAL))
        print('      Its accuracy above is an artefact of the class split -- read balanced '
              'accuracy, not accuracy.')
    print(classification_report(r['labels'], r['preds'], labels=list(range(len(r['target_names']))),
                                target_names=r['target_names'], zero_division=0))

    cm = r['confusion_matrix']
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    names = r['target_names']
    n = len(names)
    ax.set(xticks=range(n), yticks=range(n), xticklabels=names, yticklabels=names,
           xlabel='Predicted', ylabel='True', title=f'{r["name"][:40]}')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    for i in range(n):
        for j in range(n):
            v = cm[i, j]
            ax.text(j, i, str(v), ha='center', va='center',
                    color='white' if v > cm.max() / 2 else 'black')
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------------------------
# OFFICIAL (zero-shot) vs FINE-TUNED, same test clips, call-vs-noise.
#
# REVIEW FIX (Space Bunny Free): three things were wrong with this table.
#  1. It dropped the `kind` column, so the official CLASSIFIER's row read
#     "official official_classifier_m09  acc 0.8069" with nothing to say that 0.8069 is
#     117/145 -- the model answers "not noise" for every single clip -- and that its AUC is
#     0.4924, i.e. no signal at all. That is exactly the misreading §8.2 item 4 was written
#     to prevent, reintroduced one cell later for the one model where it bites hardest.
#     Fixed: `kind` and a NO SIGNAL marker are now shown, and chance-level rows are called out.
#  2. It prefixed "official " to a name that already began with "official_", printing
#     "official official_detector_m03".
#  3. It lumped full-split (145 clip) rows together with reduced-denominator ones.
# ------------------------------------------------------------------------------
print('\n' + '='*96)
print('OFFICIAL (zero-shot) vs FINE-TUNED -- call vs noise, same test clips, same windowing')
print('='*96)
print(f'{"Model":<30} {"n":>4} {"Acc":>7} {"Bal acc":>8} {"AUC":>7}  Note')
print('-'*96)


def _signal_note(r):
    notes = [r['kind']]
    if r.get('no_signal'):
        notes.append('NO SIGNAL (AUC~0.5)')
    if r['n_files'] != len(test_wavs):
        notes.append(f'only {r["n_files"]}/{len(test_wavs)} clips')
    return '; '.join(notes)


for r in val_results:
    if 'error' in r or r['auc'] is None:
        continue
    tag = model_tags.get(r['path'], '')
    label = (tag or r['name']).replace('.pk', '')[:30]
    print(f'{label:<30} {r["n_files"]:>4} {r["accuracy"]:>7.4f} '
          f'{r["balanced_accuracy"]:>8.4f} {r["auc"]:>7.4f}  {_signal_note(r)}')
for _mic, _dr in det_results.items():
    _m = _dr['metrics']
    # FIX: roc_auc_score raises on a single-class y_true, which is reachable with
    # SPLIT_BY_RECORDING=True (heti vanishes from a fold entirely). Guard it like Cell 17 does.
    _y = _m['labels']
    _auc = (roc_auc_score(_y, _m['probs'][:, 1]) if len(set(_y)) > 1 else float('nan'))
    print(f'{("fine-tuned detector " + _mic)[:30]:<30} {len(_y):>4} {_m["accuracy"]:>7.4f} '
          f'{_m["balanced_accuracy"]:>8.4f} {_auc:>7.4f}  fine-tuned, in-split')
print('-'*96)
print('Fine-tuning gain over the best official detector, BALANCED ACCURACY (the fair comparison):')
_off = [r['balanced_accuracy'] for r in val_results
        if 'error' not in r and r['auc'] is not None and r['kind'].startswith('detector')]
_fine = [_dr['metrics']['balanced_accuracy'] for _dr in det_results.values()]
if _off and _fine:
    print(f'  official best {max(_off):.4f} -> fine-tuned best {max(_fine):.4f}  '
          f'(+{max(_fine) - max(_off):.4f})')
    print('  Plain ACCURACY overstates this: the official detectors over-predict "call" on this')
    print('  81/19 split (noise precision 0.33-0.39), which costs them accuracy while leaving')
    print('  their ranking (AUC) intact. Balanced accuracy and AUC are the comparable columns.')


In [ ]:
# Cell 19: Example Predictions from Each Model (random test clips)

for r in val_results:
    if 'error' in r:
        continue
    names = r['target_names']
    print(f'\n--- {r["name"]} ({r["kind"]}) ---')
    print(f'Classes: {names}')

    n_show = min(8, len(r['preds']))
    indices = np.random.choice(len(r['preds']), n_show, replace=False)
    for idx in sorted(indices):
        true_cls = names[r['labels'][idx]] if r['labels'][idx] < len(names) else '?'
        pred_cls = names[r['preds'][idx]] if r['preds'][idx] < len(names) else '?'
        fname = os.path.basename(r['files'][idx])
        match = 'OK' if r['labels'][idx] == r['preds'][idx] else 'MISS'
        print(f'  {match} true={true_cls:6s} pred={pred_cls:6s}  {fname}')

## Extra evidence — added by Space Bunny Free

Three cells were appended to this notebook. They do not change the main pipeline; they
produce the evidence needed to *interpret* it.

| Cell | What it does | Why |
| :--- | :--- | :--- |
| **20** | Classifier A/B for `USE_AUGMENTATION`, then the same configuration over seeds 42/43/44 | Every number so far comes from **one** 145-clip split. The run-to-run spread is ~±2 points, which is the size of most differences being argued about. Each seed re-splits *and* re-initialises, so the sd is total spread. |
| **21** | Retrains the classifier and the best mic with **whole tapes held out** | Cell 7 reports 145/145 test clips share a recording with train, across only 28 recordings. No arithmetic converts that into a new-recording number — the only honest estimate requires holding recordings out. |

### How to read the result

The in-split scores in Cells 12–18 are an **upper bound**. They are fine for comparing models
*against each other on the same split*. They are not a forecast for a new recording.

Two structural limits are worth stating plainly, because no amount of tuning moves them:

1. **28 recordings is the binding constraint.** The windowing fix multiplies distinct training
   *inputs* by ~×56 (classifier) and ~×43 (detector), but all ~56 windows of a clip come from
   the same tape and are near-duplicates. For generalisation the effective sample size is still
   28 recordings.
2. **The classes with perfect scores are the ones with the fewest recordings** — heti (1 tape),
   rhbe (~2) — while every class with many tapes carries the errors. That is the signature of
   leakage, and it is why the grouped number is usually lower.

The honest next step for accuracy is more recordings of `rhro`, `rhle`, and the
`acsh`↔`alte` pair (which also deserves a label review — both are *Myotis* with near-identical
~40–45 kHz FM sweeps, and 10 of the 21 remaining errors are false `acsh`), not another sweep of
the learning rate.


In [ ]:
# Cell 20: Classifier ablations — augmentation A/B and multi-seed spread
#
# Runs AFTER the main models are trained, evaluated, exported and summarised, and it uses its
# own variable names throughout, so it cannot disturb det_results / cls_model / pipeline_results
# / val_results. Everything here is controlled by RUN_ABLATIONS in Cell 2.
#
# WHY (Space Bunny Free)
# 1. USE_AUGMENTATION has been toggled by hand and argued about in the notes, but never A/B
#    tested on the windowed dataset. Random window cropping is now the dominant augmenter
#    (x56 distinct inputs), so the extra transforms may be redundant or may hurt. One run each
#    way settles it.
# 2. Every number in this notebook comes from ONE 145-clip test split. The measured run-to-run
#    spread is about +/-2 points, which is the same size as most of the differences being
#    discussed (cascade variants, m03 vs from-scratch, top-1 vs top-5 windows). Until the
#    spread is on the page, none of those differences can be believed. Each seed here
#    RE-SPLITS and RE-INITIALISES, so the reported sd is total run-to-run spread -- the
#    conservative number, and the one that matters for deciding whether a delta is real.

if RUN_ABLATIONS:
    def _abl_split(seed):
        """Same 70/15/15 recipe as Cell 7, but with a variable seed."""
        tr, te, trl, _ = train_test_split(all_wavs, all_species_labels, test_size=0.15,
                                          random_state=seed, stratify=all_species_labels)
        tr, va, _, _ = train_test_split(tr, trl, test_size=0.15 / 0.85, random_state=seed,
                                        stratify=trl)
        return tr, va, te

    def _abl_fresh_classifier(seed):
        """Fresh classifier head + the official encoder, with a reproducible init."""
        torch.manual_seed(seed)
        np.random.seed(seed)
        _random.seed(seed)
        model, enc, _head, _opts = build_model(encoderOpts_cls, CLS_CONFIG['num_classes'], DEVICE)
        if classifier_path:
            try:
                _pre, _, _ = load_model_from_pk(classifier_path, DEVICE)
                enc.load_state_dict(dict(list(_pre.named_children())[0][1].state_dict()))
            except Exception as _e:                        # noqa: BLE001
                print(f'  [ablation] could not load pretrained encoder: {_e}')
        return model.to(DEVICE)

    def _abl_classifier_run(seed, augment):
        tr, va, te = _abl_split(seed)
        d_tr = WindowedBatDataset(tr, CLS_CLASS_TO_IDX, CLS_CONFIG, train=True, augment=augment)
        d_va = WindowedBatDataset(va, CLS_CLASS_TO_IDX, CLS_CONFIG, train=False)
        d_te = WindowedBatDataset(te, CLS_CLASS_TO_IDX, CLS_CONFIG, train=False)
        model = _abl_fresh_classifier(seed)
        model, hist, best = train_model(
            model, d_tr, d_va, CLS_CONFIG, DEVICE,
            save_path=os.path.join(WORKING_DIR, f'ablation_cls_s{seed}_aug{int(augment)}.png'))
        m = evaluate_model(model, d_te, CLS_CLASSES, DEVICE,
                           f'Ablation classifier seed={seed} augment={augment}')
        return {'seed': seed, 'augment': augment,
                'test_acc': m['accuracy'], 'test_bal_acc': m['balanced_accuracy'],
                'test_f1': m['f1'], 'best_val': best, 'best_epoch': hist['best_epoch'],
                'topk_mean': hist['topk_mean'],
                'n_train_clips': len(tr), 'n_test_clips': len(te)}

    print('\n' + '=' * 88)
    print(f'ABLATION 1 -- USE_AUGMENTATION on/off (seed {ABLATION_SEEDS[0]})')
    print('=' * 88)
    _aug_runs = {}
    for _aug in (False, True):
        print(f'\n>>> augment={_aug}')
        _aug_runs[_aug] = _abl_classifier_run(ABLATION_SEEDS[0], _aug)
        print(f'  [ablation] augment={_aug}: test acc={_aug_runs[_aug]["test_acc"]:.4f} '
              f'bal_acc={_aug_runs[_aug]["test_bal_acc"]:.4f}')

    _d_aug = _aug_runs[True]['test_bal_acc'] - _aug_runs[False]['test_bal_acc']
    print(f'\n  augmentation delta (balanced acc): {_d_aug:+.4f}')
    print('  Read against the seed spread below before concluding anything.')

    print('\n' + '=' * 88)
    print(f'ABLATION 2 -- seed spread, no augmentation (seeds {ABLATION_SEEDS})')
    print('=' * 88)
    _seed_runs = []
    for _s in ABLATION_SEEDS:
        print(f'\n>>> seed {_s}')
        _r = _abl_classifier_run(_s, False)
        _seed_runs.append(_r)
        print(f'  [ablation] seed {_s}: test acc={_r["test_acc"]:.4f} '
              f'bal_acc={_r["test_bal_acc"]:.4f} (best epoch {_r["best_epoch"]})')

    _accs = np.array([r['test_acc'] for r in _seed_runs])
    _bals = np.array([r['test_bal_acc'] for r in _seed_runs])
    print('\n' + '-' * 88)
    print(f'{"seed":<6} {"test acc":>10} {"test bal acc":>14} {"best val":>10} {"best ep":>9}')
    for r in _seed_runs:
        print(f'{r["seed"]:<6} {r["test_acc"]:>10.4f} {r["test_bal_acc"]:>14.4f} '
              f'{r["best_val"]:>10.4f} {str(r["best_epoch"]):>9}')
    print('-' * 88)
    print(f'{"mean":<6} {_accs.mean():>10.4f} {_bals.mean():>14.4f}')
    if len(_accs) > 1:
        print(f'{"sd":<6} {_accs.std(ddof=1):>10.4f} {_bals.std(ddof=1):>14.4f}')
        print(f'\n  => differences smaller than about {2 * _bals.std(ddof=1):.3f} balanced '
              f'accuracy (2 sd) are NOT distinguishable from run-to-run noise in this setup.')
        print('     Any earlier claim resting on a 1-3 point gap should be re-read with this '
              'number beside it.')
    else:
        print('  (single seed: set ABLATION_SEEDS to 2+ values in Cell 2 for a spread)')
    ablation_results = {'augment': _aug_runs, 'seeds': _seed_runs,
                        'mean_acc': float(_accs.mean()), 'mean_bal_acc': float(_bals.mean()),
                        'sd_acc': float(_accs.std(ddof=1)) if len(_accs) > 1 else None,
                        'sd_bal_acc': float(_bals.std(ddof=1)) if len(_bals) > 1 else None}
else:
    print('\n[ablations skipped: RUN_ABLATIONS = False]')
    ablation_results = None


In [ ]:
# Cell 21: Recording-grouped check -- the only honest "new recording" estimate
#
# Controlled by RUN_GROUPED_CHECK / GROUPED_MICS in Cell 2. Uses its own variable names and
# runs last, so the in-split results above are untouched.
#
# WHY THIS CELL EXISTS (Space Bunny Free)
# Cell 7 reports that 145/145 test clips come from a recording that ALSO appears in train, and
# that there are only 28 recordings for 964 clips -- with heti coming from a single tape and
# 86 of 106 rhbe clips from one. The classifier's per-class results line up with that exactly:
# the classes recorded once or twice (heti 10/10, rhbe 16/16, sasa 14/14) are perfect, while
# the classes with many recordings carry the errors (rhro recall 0.57, acsh precision 0.68).
#
# That pattern cannot be turned into a new-recording number by arithmetic. There is no clean
# subset of the existing test split to score, because every one of its clips shares a recording
# with train. The only way to measure generalisation to a new tape is to RETRAIN with whole
# tapes held out. That is what this cell does.
#
# The windowing fix does not help here, and it is worth being explicit about why: it multiplies
# the number of distinct training INPUTS by ~x56, but all ~56 windows of a clip come from the
# same tape and are near-duplicates of one another. For generalisation the effective sample
# size is still 28 recordings, and that -- not the model, not the optimiser, not the learning
# rate -- is what now limits accuracy.

if RUN_GROUPED_CHECK:
    from sklearn.model_selection import StratifiedGroupKFold

    _groups = [recording_id(w) for w in all_wavs]
    _n_splits = 7
    _folds = [te for _, te in StratifiedGroupKFold(n_splits=_n_splits, shuffle=True,
                                                   random_state=1)
              .split(all_wavs, all_species_labels, _groups)]
    g_test = [all_wavs[i] for i in _folds[0]]
    g_val = [all_wavs[i] for i in _folds[1]]
    _held = set(_folds[0]) | set(_folds[1])
    g_train = [w for i, w in enumerate(all_wavs) if i not in _held]

    g_test_species = {get_class_from_filename(w) for w in g_test}
    g_train_species = {get_class_from_filename(w) for w in g_train}
    print('\n' + '=' * 88)
    print(f'RECORDING-GROUPED SPLIT — whole tapes held out ({_n_splits} folds of '
          f'{len(all_rec_groups)} recordings)')
    print('=' * 88)
    print(f'  train={len(g_train)} clips / {len({recording_id(w) for w in g_train})} recordings')
    print(f'  val  ={len(g_val)} clips / {len({recording_id(w) for w in g_val})} recordings')
    print(f'  test ={len(g_test)} clips / {len({recording_id(w) for w in g_test})} recordings')
    print(f'  classes in test : {sorted(g_test_species)}')
    if g_test_species != g_train_species:
        print(f'  WARNING: classes absent from train: {sorted(g_test_species - g_train_species)}; '
              f'classes absent from test: {sorted(g_train_species - g_test_species)}')
        print('           A class missing from the test fold simply cannot be scored, and the '
              'balanced accuracy below averages over fewer classes than 8.')
    _ov = len({recording_id(w) for w in g_test} & {recording_id(w) for w in g_train})
    print(f'  recordings shared between train and test: {_ov}  (must be 0)')

    grouped_results = {}

    # ---- classifier ------------------------------------------------------------------------
    print(f'\n{ "-" * 88 }\nGROUPED classifier (250 kHz)\n{ "-" * 88 }')
    g_cls_tr = WindowedBatDataset(g_train, CLS_CLASS_TO_IDX, CLS_CONFIG, train=True,
                                  augment=USE_AUGMENTATION)
    g_cls_va = WindowedBatDataset(g_val, CLS_CLASS_TO_IDX, CLS_CONFIG, train=False)
    g_cls_te = WindowedBatDataset(g_test, CLS_CLASS_TO_IDX, CLS_CONFIG, train=False)
    torch.manual_seed(42); np.random.seed(42); _random.seed(42)
    g_cls_model, g_cls_enc, _h, _o = build_model(encoderOpts_cls, CLS_CONFIG['num_classes'], DEVICE)
    if classifier_path:
        try:
            _pre, _, _ = load_model_from_pk(classifier_path, DEVICE)
            g_cls_enc.load_state_dict(dict(list(_pre.named_children())[0][1].state_dict()))
            print(f'  encoder loaded from {os.path.basename(classifier_path)}')
        except Exception as _e:
            print(f'  could not load pretrained encoder: {_e}')
    g_cls_model, g_cls_hist, g_cls_best = train_model(
        g_cls_model.to(DEVICE), g_cls_tr, g_cls_va, CLS_CONFIG, DEVICE,
        save_path=os.path.join(WORKING_DIR, 'grouped_classifier_curves.png'))
    g_cls_m = evaluate_model(g_cls_model, g_cls_te, CLS_CLASSES, DEVICE,
                             'GROUPED classifier (unseen recordings)')
    grouped_results['classifier'] = g_cls_m

    # ---- best mic(s) only --------------------------------------------------------------------
    for _mic in GROUPED_MICS:
        print(f'\n{ "-" * 88 }\nGROUPED detector {_mic} (192 kHz)\n{ "-" * 88 }')
        _pk = PRETRAINED_MODELS.get('detector', {}).get(_mic, {}).get('path')
        det_class_to_idx = {'noise': 0}
        det_class_to_idx.update({c: 1 for c in CLS_CLASSES if c != 'noise'})
        g_det_tr = WindowedBatDataset(g_train, det_class_to_idx, DET_CONFIG, train=True,
                                      augment=USE_AUGMENTATION)
        g_det_va = WindowedBatDataset(g_val, det_class_to_idx, DET_CONFIG, train=False)
        g_det_te = WindowedBatDataset(g_test, det_class_to_idx, DET_CONFIG, train=False)
        torch.manual_seed(42); np.random.seed(42); _random.seed(42)
        g_m, g_enc, _h2, _o2 = build_model(encoderOpts_det, DET_CONFIG['num_classes'], DEVICE)
        if _pk:
            try:
                _pre, _, _ = load_model_from_pk(_pk, DEVICE)
                g_enc.load_state_dict(dict(list(_pre.named_children())[0][1].state_dict()))
                print(f'  encoder loaded from {os.path.basename(_pk)}')
            except Exception as _e:
                print(f'  could not load pretrained encoder: {_e}')
        g_m, g_hist, g_best = train_model(
            g_m.to(DEVICE), g_det_tr, g_det_va, DET_CONFIG, DEVICE,
            save_path=os.path.join(WORKING_DIR, f'grouped_detector_{_mic}_curves.png'))
        g_m_m = evaluate_model(g_m, g_det_te, DET_CLASSES, DEVICE,
                               f'GROUPED detector {_mic} (unseen recordings)')
        grouped_results[f'detector_{_mic}'] = g_m_m

    # ---- side-by-side ------------------------------------------------------------------------
    print('\n' + '=' * 88)
    print('IN-SPLIT vs RECORDING-GROUPED  (the gap is the leakage)')
    print('=' * 88)
    print(f'{"Task":<22} {"in-split acc":>13} {"in-split bal":>13} | '
          f'{"grouped acc":>12} {"grouped bal":>12}')
    print('-' * 88)
    _rows = [('classifier', cls_metrics, grouped_results.get('classifier'))]
    for _mic in GROUPED_MICS:
        if f'detector_{_mic}' in grouped_results:
            _rows.append((f'detector {_mic}', det_results[_mic]['metrics'],
                          grouped_results[f'detector_{_mic}']))
    for _name, _ins, _grp in _rows:
        if _grp is None:
            continue
        print(f'{_name:<22} {_ins["accuracy"]:>13.4f} {_ins["balanced_accuracy"]:>13.4f} | '
              f'{_grp["accuracy"]:>12.4f} {_grp["balanced_accuracy"]:>12.4f}')
    print('-' * 88)
    print('The in-split column is an UPPER BOUND. Quote the grouped column for any claim about')
    print('performance on a recording the model has never seen. Note also that a grouped split')
    print(f'over {_n_splits} folds is noisier than the 145-clip in-split test set, and classes')
    print('recorded once cannot be assessed at all.')
else:
    print('\n[grouped check skipped: RUN_GROUPED_CHECK = False]')
    grouped_results = None
